# WESAD | Análisis univariado y bivariado de 455 características — entrenamiento exclusivamente

**Objetivo.** Caracterizar cada variable fisiológica y su relación descriptiva con el objetivo binario (`target=1`: estrés; `target=0`: no estrés), a partir de **los diez participantes S2–S11 en conjunto**. Las figuras, estadísticas y pruebas se generan automáticamente **para cada característica**, con códigos concordantes con la tesis.

**Separación definitiva:** los sujetos **S13–S17 constituyen el conjunto externo de prueba**. Este notebook **no abre, inspecciona, grafica ni utiliza** sus archivos. La reserva evita decisiones guiadas por distribuciones, valores faltantes o asociaciones presentes en evaluación externa. Un futuro ajuste de hiperparámetros o selección supervisada necesita validación interna por sujeto dentro de los diez participantes.

> **Unidad independiente: el participante, no la ventana.** Las ventanas de 60 s desplazadas 250 ms se solapan un 99,58 %. Las estadísticas por ventana describen el conjunto, pero un p valor calculado como si todas las ventanas fueran independientes sería seudorreplicación. Por eso la prueba principal usa *medianas por sujeto y clase* (hasta diez pares) y una prueba exacta de inversión de signos; los histogramas, violines y boxplots se muestran con las ventanas de los diez participantes.

**Significado del negativo:** `target=0` agrupa las condiciones originales `baseline` (1) y `amusement` (3); `target=1` representa `stress` (2). La inferencia compara estrés frente a **la mezcla** de las dos condiciones negativas, no solamente frente al estado basal. Las ventanas con otras etiquetas fueron excluidas durante la extracción original.

## Diseño de memoria: lectura una sola vez por sujeto

Los checkpoints `*_features_full.pkl` están serializados con `pickle`: **no permiten leer una sola columna del archivo sin deserializar la matriz completa**. Por tanto, no sería realista afirmar que abrimos directamente la columna ECG_1 en el `.pkl`. Para proteger equipos de 16 GB:

1. **Conversión inicial en streaming por sujeto:** leer **un archivo completo a la vez**, extraer únicamente las 455 columnas ECG/EDA/RESP y grabarlas en una matriz `memmap` en disco con organización **característica × ventanas**. Se liberan los objetos y se ejecuta `gc.collect()` antes de abrir el siguiente sujeto. No se procesan los sujetos externos.
2. **Análisis posterior:** cargar mediante mapeo de memoria **una sola característica de los diez sujetos**, más sus vectores target. Los datos de las demás características no entran en RAM. Se descartan valores no finitos para las estadísticas, informando proporciones.
3. **Gráficos:** para las nubes/histogramas se usa una submuestra reproducible por sujeto y clase (disminuye tiempo y congestión visual), pero los estadísticos descriptivos se obtienen con **todas las ventanas finitas de entrenamiento**. Las pruebas usan agregados por sujeto.

La caché puede ocupar **varios GB en disco**; usa espacio libre suficiente. Cada checkpoint completo podría ocupar una cantidad considerable de RAM *durante su lectura inicial*; Python no puede abrir una sola columna de un pickle comprimido o convencional. Si una sola matriz supera el margen de memoria disponible, será necesario convertirla desde la máquina donde se creó, o exportar las columnas en origen.

In [1]:
from pathlib import Path
import os, gc, pickle, json, math, time, warnings
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from scipy.stats import wasserstein_distance, rankdata
from IPython.display import display

# Sujeto S2 a S11 = DIEZ sujetos de entrenamiento; NO acceder a S13–S17.
TRAIN_SUBJECTS = [f'S{i}' for i in range(2, 12)]
TEST_RESERVED = [f'S{i}' for i in range(13, 18)]
BASE_DIR = Path.cwd()  # ejecutar desde THESIS_WESAD; cambiar si procede
FULL_DIR = BASE_DIR / 'wesad_feature_output' / 'checkpoints_completos'
OUT_DIR = BASE_DIR / 'wesad_feature_output' / 'Analisis_univariado_bivariado_train_10_sujetos'
CACHE_DIR = OUT_DIR / 'cache_memmap_train'
FIG_DIR = OUT_DIR / 'figuras_455'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DTYPE = np.float32
CACHE_BLOCK_FEATURES = 8
MAX_POINTS_PER_SUBJECT_CLASS = 1200  # representación visual, NO cálculo de estadísticas
EXPORT_ALL_FIGURES = True
EXPORT_DPI = 220
SHOW_FIGURES_IN_NOTEBOOK = False  # False evita acumular 455 figuras en memoria y notebook
FIGURE_LIMIT = None  # None: revisar y completar las 455
RESUME_FIGURES = True  # omite PNGs existentes y validados
REUSE_STATS_CSV = True  # reutiliza los 3 CSV completos si son compatibles
SEED = 2026
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)
plt.rcParams.update({'font.size': 15, 'axes.titlesize': 16, 'axes.labelsize': 14, 'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 12, 'figure.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})
print('Sujetos ENTRENAMIENTO:', TRAIN_SUBJECTS)
print('Sujetos RESERVADOS SIN ACCESO:', TEST_RESERVED)
print('Checkpoints:', FULL_DIR)

Sujetos ENTRENAMIENTO: ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11']
Sujetos RESERVADOS SIN ACCESO: ['S13', 'S14', 'S15', 'S16', 'S17']
Checkpoints: C:\Users\Victor\Documents\THESIS_WESAD\wesad_feature_output\checkpoints_completos


## Catálogo de nombres y códigos **de la tesis**

El identificador no se reconstruye ordenando alfabéticamente ni confiando en la numeración impresa por el notebook de verificación. Se incorpora una tabla `nombre exacto → código de la tesis` extraída de las tablas del documento de maestría, con `EDA_225` y `EDA_226` añadidas. La correspondencia se verifica contra los 455 nombres de los checkpoints, informando cualquier discrepancia. El código del notebook y el código de la tesis pueden diferir en su orden y eso no es un error de extracción.

In [2]:
CATALOGO = pd.DataFrame(json.loads(r'''[{"senal": "ECG", "codigo_tesis": "ECG_15", "feature_name": "ecg_raw_mean", "orden_checkpoint": 1}, {"senal": "ECG", "codigo_tesis": "ECG_20", "feature_name": "ecg_raw_median", "orden_checkpoint": 2}, {"senal": "ECG", "codigo_tesis": "ECG_49", "feature_name": "ecg_raw_std", "orden_checkpoint": 3}, {"senal": "ECG", "codigo_tesis": "ECG_51", "feature_name": "ecg_raw_variance", "orden_checkpoint": 4}, {"senal": "ECG", "codigo_tesis": "ECG_22", "feature_name": "ecg_raw_min", "orden_checkpoint": 5}, {"senal": "ECG", "codigo_tesis": "ECG_14", "feature_name": "ecg_raw_max", "orden_checkpoint": 6}, {"senal": "ECG", "codigo_tesis": "ECG_35", "feature_name": "ecg_raw_range", "orden_checkpoint": 7}, {"senal": "ECG", "codigo_tesis": "ECG_27", "feature_name": "ecg_raw_q01", "orden_checkpoint": 8}, {"senal": "ECG", "codigo_tesis": "ECG_28", "feature_name": "ecg_raw_q05", "orden_checkpoint": 9}, {"senal": "ECG", "codigo_tesis": "ECG_29", "feature_name": "ecg_raw_q10", "orden_checkpoint": 10}, {"senal": "ECG", "codigo_tesis": "ECG_30", "feature_name": "ecg_raw_q25", "orden_checkpoint": 11}, {"senal": "ECG", "codigo_tesis": "ECG_31", "feature_name": "ecg_raw_q75", "orden_checkpoint": 12}, {"senal": "ECG", "codigo_tesis": "ECG_32", "feature_name": "ecg_raw_q90", "orden_checkpoint": 13}, {"senal": "ECG", "codigo_tesis": "ECG_33", "feature_name": "ecg_raw_q95", "orden_checkpoint": 14}, {"senal": "ECG", "codigo_tesis": "ECG_34", "feature_name": "ecg_raw_q99", "orden_checkpoint": 15}, {"senal": "ECG", "codigo_tesis": "ECG_10", "feature_name": "ecg_raw_iqr", "orden_checkpoint": 16}, {"senal": "ECG", "codigo_tesis": "ECG_12", "feature_name": "ecg_raw_mad_mean", "orden_checkpoint": 17}, {"senal": "ECG", "codigo_tesis": "ECG_13", "feature_name": "ecg_raw_mad_median", "orden_checkpoint": 18}, {"senal": "ECG", "codigo_tesis": "ECG_40", "feature_name": "ecg_raw_rms", "orden_checkpoint": 19}, {"senal": "ECG", "codigo_tesis": "ECG_16", "feature_name": "ecg_raw_mean_absolute_value", "orden_checkpoint": 20}, {"senal": "ECG", "codigo_tesis": "ECG_8", "feature_name": "ecg_raw_energy", "orden_checkpoint": 21}, {"senal": "ECG", "codigo_tesis": "ECG_19", "feature_name": "ecg_raw_mean_power", "orden_checkpoint": 22}, {"senal": "ECG", "codigo_tesis": "ECG_1", "feature_name": "ecg_raw_auc_absolute", "orden_checkpoint": 23}, {"senal": "ECG", "codigo_tesis": "ECG_2", "feature_name": "ecg_raw_auc_signed", "orden_checkpoint": 24}, {"senal": "ECG", "codigo_tesis": "ECG_42", "feature_name": "ecg_raw_skewness", "orden_checkpoint": 25}, {"senal": "ECG", "codigo_tesis": "ECG_11", "feature_name": "ecg_raw_kurtosis", "orden_checkpoint": 26}, {"senal": "ECG", "codigo_tesis": "ECG_43", "feature_name": "ecg_raw_slope", "orden_checkpoint": 27}, {"senal": "ECG", "codigo_tesis": "ECG_6", "feature_name": "ecg_raw_delta", "orden_checkpoint": 28}, {"senal": "ECG", "codigo_tesis": "ECG_4", "feature_name": "ecg_raw_coefficient_variation", "orden_checkpoint": 29}, {"senal": "ECG", "codigo_tesis": "ECG_78", "feature_name": "ecg_raw_zero_crossing_rate", "orden_checkpoint": 30}, {"senal": "ECG", "codigo_tesis": "ECG_17", "feature_name": "ecg_raw_mean_crossing_rate", "orden_checkpoint": 31}, {"senal": "ECG", "codigo_tesis": "ECG_52", "feature_name": "ecg_raw_waveform_length", "orden_checkpoint": 32}, {"senal": "ECG", "codigo_tesis": "ECG_5", "feature_name": "ecg_raw_crest_factor", "orden_checkpoint": 33}, {"senal": "ECG", "codigo_tesis": "ECG_41", "feature_name": "ecg_raw_shape_factor", "orden_checkpoint": 34}, {"senal": "ECG", "codigo_tesis": "ECG_9", "feature_name": "ecg_raw_impulse_factor", "orden_checkpoint": 35}, {"senal": "ECG", "codigo_tesis": "ECG_3", "feature_name": "ecg_raw_clearance_factor", "orden_checkpoint": 36}, {"senal": "ECG", "codigo_tesis": "ECG_7", "feature_name": "ecg_raw_dominant_frequency", "orden_checkpoint": 37}, {"senal": "ECG", "codigo_tesis": "ECG_18", "feature_name": "ecg_raw_mean_frequency", "orden_checkpoint": 38}, {"senal": "ECG", "codigo_tesis": "ECG_21", "feature_name": "ecg_raw_median_frequency", "orden_checkpoint": 39}, {"senal": "ECG", "codigo_tesis": "ECG_45", "feature_name": "ecg_raw_spectral_entropy", "orden_checkpoint": 40}, {"senal": "ECG", "codigo_tesis": "ECG_50", "feature_name": "ecg_raw_total_spectral_power", "orden_checkpoint": 41}, {"senal": "ECG", "codigo_tesis": "ECG_44", "feature_name": "ecg_raw_spectral_centroid", "orden_checkpoint": 42}, {"senal": "ECG", "codigo_tesis": "ECG_48", "feature_name": "ecg_raw_spectral_spread", "orden_checkpoint": 43}, {"senal": "ECG", "codigo_tesis": "ECG_47", "feature_name": "ecg_raw_spectral_skewness", "orden_checkpoint": 44}, {"senal": "ECG", "codigo_tesis": "ECG_46", "feature_name": "ecg_raw_spectral_kurtosis", "orden_checkpoint": 45}, {"senal": "ECG", "codigo_tesis": "ECG_26", "feature_name": "ecg_raw_power_very_low", "orden_checkpoint": 46}, {"senal": "ECG", "codigo_tesis": "ECG_39", "feature_name": "ecg_raw_relative_power_very_low", "orden_checkpoint": 47}, {"senal": "ECG", "codigo_tesis": "ECG_24", "feature_name": "ecg_raw_power_low", "orden_checkpoint": 48}, {"senal": "ECG", "codigo_tesis": "ECG_37", "feature_name": "ecg_raw_relative_power_low", "orden_checkpoint": 49}, {"senal": "ECG", "codigo_tesis": "ECG_25", "feature_name": "ecg_raw_power_mid", "orden_checkpoint": 50}, {"senal": "ECG", "codigo_tesis": "ECG_38", "feature_name": "ecg_raw_relative_power_mid", "orden_checkpoint": 51}, {"senal": "ECG", "codigo_tesis": "ECG_23", "feature_name": "ecg_raw_power_high", "orden_checkpoint": 52}, {"senal": "ECG", "codigo_tesis": "ECG_36", "feature_name": "ecg_raw_relative_power_high", "orden_checkpoint": 53}, {"senal": "ECG", "codigo_tesis": "ECG_53", "feature_name": "ecg_raw_wavelet_A_energy", "orden_checkpoint": 54}, {"senal": "ECG", "codigo_tesis": "ECG_56", "feature_name": "ecg_raw_wavelet_A_relative_energy", "orden_checkpoint": 55}, {"senal": "ECG", "codigo_tesis": "ECG_55", "feature_name": "ecg_raw_wavelet_A_mean_abs", "orden_checkpoint": 56}, {"senal": "ECG", "codigo_tesis": "ECG_57", "feature_name": "ecg_raw_wavelet_A_std", "orden_checkpoint": 57}, {"senal": "ECG", "codigo_tesis": "ECG_54", "feature_name": "ecg_raw_wavelet_A_entropy", "orden_checkpoint": 58}, {"senal": "ECG", "codigo_tesis": "ECG_73", "feature_name": "ecg_raw_wavelet_D4_energy", "orden_checkpoint": 59}, {"senal": "ECG", "codigo_tesis": "ECG_76", "feature_name": "ecg_raw_wavelet_D4_relative_energy", "orden_checkpoint": 60}, {"senal": "ECG", "codigo_tesis": "ECG_75", "feature_name": "ecg_raw_wavelet_D4_mean_abs", "orden_checkpoint": 61}, {"senal": "ECG", "codigo_tesis": "ECG_77", "feature_name": "ecg_raw_wavelet_D4_std", "orden_checkpoint": 62}, {"senal": "ECG", "codigo_tesis": "ECG_74", "feature_name": "ecg_raw_wavelet_D4_entropy", "orden_checkpoint": 63}, {"senal": "ECG", "codigo_tesis": "ECG_68", "feature_name": "ecg_raw_wavelet_D3_energy", "orden_checkpoint": 64}, {"senal": "ECG", "codigo_tesis": "ECG_71", "feature_name": "ecg_raw_wavelet_D3_relative_energy", "orden_checkpoint": 65}, {"senal": "ECG", "codigo_tesis": "ECG_70", "feature_name": "ecg_raw_wavelet_D3_mean_abs", "orden_checkpoint": 66}, {"senal": "ECG", "codigo_tesis": "ECG_72", "feature_name": "ecg_raw_wavelet_D3_std", "orden_checkpoint": 67}, {"senal": "ECG", "codigo_tesis": "ECG_69", "feature_name": "ecg_raw_wavelet_D3_entropy", "orden_checkpoint": 68}, {"senal": "ECG", "codigo_tesis": "ECG_63", "feature_name": "ecg_raw_wavelet_D2_energy", "orden_checkpoint": 69}, {"senal": "ECG", "codigo_tesis": "ECG_66", "feature_name": "ecg_raw_wavelet_D2_relative_energy", "orden_checkpoint": 70}, {"senal": "ECG", "codigo_tesis": "ECG_65", "feature_name": "ecg_raw_wavelet_D2_mean_abs", "orden_checkpoint": 71}, {"senal": "ECG", "codigo_tesis": "ECG_67", "feature_name": "ecg_raw_wavelet_D2_std", "orden_checkpoint": 72}, {"senal": "ECG", "codigo_tesis": "ECG_64", "feature_name": "ecg_raw_wavelet_D2_entropy", "orden_checkpoint": 73}, {"senal": "ECG", "codigo_tesis": "ECG_58", "feature_name": "ecg_raw_wavelet_D1_energy", "orden_checkpoint": 74}, {"senal": "ECG", "codigo_tesis": "ECG_61", "feature_name": "ecg_raw_wavelet_D1_relative_energy", "orden_checkpoint": 75}, {"senal": "ECG", "codigo_tesis": "ECG_60", "feature_name": "ecg_raw_wavelet_D1_mean_abs", "orden_checkpoint": 76}, {"senal": "ECG", "codigo_tesis": "ECG_62", "feature_name": "ecg_raw_wavelet_D1_std", "orden_checkpoint": 77}, {"senal": "ECG", "codigo_tesis": "ECG_59", "feature_name": "ecg_raw_wavelet_D1_entropy", "orden_checkpoint": 78}, {"senal": "ECG", "codigo_tesis": "ECG_79", "feature_name": "ecg_rpeak_count", "orden_checkpoint": 79}, {"senal": "ECG", "codigo_tesis": "ECG_80", "feature_name": "ecg_rpeak_rate_per_minute", "orden_checkpoint": 80}, {"senal": "ECG", "codigo_tesis": "ECG_81", "feature_name": "ecg_hr_mean", "orden_checkpoint": 81}, {"senal": "ECG", "codigo_tesis": "ECG_82", "feature_name": "ecg_hr_median", "orden_checkpoint": 82}, {"senal": "ECG", "codigo_tesis": "ECG_83", "feature_name": "ecg_hr_std", "orden_checkpoint": 83}, {"senal": "ECG", "codigo_tesis": "ECG_84", "feature_name": "ecg_hr_min", "orden_checkpoint": 84}, {"senal": "ECG", "codigo_tesis": "ECG_85", "feature_name": "ecg_hr_max", "orden_checkpoint": 85}, {"senal": "ECG", "codigo_tesis": "ECG_86", "feature_name": "ecg_hr_range", "orden_checkpoint": 86}, {"senal": "ECG", "codigo_tesis": "ECG_87", "feature_name": "ecg_mean_nn", "orden_checkpoint": 87}, {"senal": "ECG", "codigo_tesis": "ECG_88", "feature_name": "ecg_median_nn", "orden_checkpoint": 88}, {"senal": "ECG", "codigo_tesis": "ECG_89", "feature_name": "ecg_sdnn", "orden_checkpoint": 89}, {"senal": "ECG", "codigo_tesis": "ECG_90", "feature_name": "ecg_rmssd", "orden_checkpoint": 90}, {"senal": "ECG", "codigo_tesis": "ECG_91", "feature_name": "ecg_sdsd", "orden_checkpoint": 91}, {"senal": "ECG", "codigo_tesis": "ECG_92", "feature_name": "ecg_pnn20", "orden_checkpoint": 92}, {"senal": "ECG", "codigo_tesis": "ECG_93", "feature_name": "ecg_pnn50", "orden_checkpoint": 93}, {"senal": "ECG", "codigo_tesis": "ECG_94", "feature_name": "ecg_rr_range", "orden_checkpoint": 94}, {"senal": "ECG", "codigo_tesis": "ECG_95", "feature_name": "ecg_rr_iqr", "orden_checkpoint": 95}, {"senal": "ECG", "codigo_tesis": "ECG_96", "feature_name": "ecg_rr_cv", "orden_checkpoint": 96}, {"senal": "ECG", "codigo_tesis": "ECG_97", "feature_name": "ecg_rr_mad", "orden_checkpoint": 97}, {"senal": "ECG", "codigo_tesis": "ECG_98", "feature_name": "ecg_sd1", "orden_checkpoint": 98}, {"senal": "ECG", "codigo_tesis": "ECG_99", "feature_name": "ecg_sd2", "orden_checkpoint": 99}, {"senal": "ECG", "codigo_tesis": "ECG_100", "feature_name": "ecg_sd1_sd2_ratio", "orden_checkpoint": 100}, {"senal": "ECG", "codigo_tesis": "ECG_101", "feature_name": "ecg_poincare_area", "orden_checkpoint": 101}, {"senal": "ECG", "codigo_tesis": "ECG_102", "feature_name": "ecg_rr_sample_entropy", "orden_checkpoint": 102}, {"senal": "ECG", "codigo_tesis": "ECG_103", "feature_name": "ecg_rr_approximate_entropy", "orden_checkpoint": 103}, {"senal": "ECG", "codigo_tesis": "ECG_104", "feature_name": "ecg_rr_permutation_entropy", "orden_checkpoint": 104}, {"senal": "ECG", "codigo_tesis": "ECG_105", "feature_name": "ecg_rr_lempel_ziv_complexity", "orden_checkpoint": 105}, {"senal": "ECG", "codigo_tesis": "ECG_106", "feature_name": "ecg_rr_petrosian_fd", "orden_checkpoint": 106}, {"senal": "ECG", "codigo_tesis": "ECG_107", "feature_name": "ecg_rr_katz_fd", "orden_checkpoint": 107}, {"senal": "ECG", "codigo_tesis": "ECG_108", "feature_name": "ecg_rr_higuchi_fd", "orden_checkpoint": 108}, {"senal": "ECG", "codigo_tesis": "ECG_109", "feature_name": "ecg_rr_dfa", "orden_checkpoint": 109}, {"senal": "ECG", "codigo_tesis": "ECG_110", "feature_name": "ecg_rr_rqa_recurrence_rate", "orden_checkpoint": 110}, {"senal": "ECG", "codigo_tesis": "ECG_111", "feature_name": "ecg_rr_rqa_determinism", "orden_checkpoint": 111}, {"senal": "ECG", "codigo_tesis": "ECG_112", "feature_name": "ecg_rr_rqa_laminarity", "orden_checkpoint": 112}, {"senal": "ECG", "codigo_tesis": "ECG_113", "feature_name": "ecg_rr_rqa_diagonal_entropy", "orden_checkpoint": 113}, {"senal": "ECG", "codigo_tesis": "ECG_114", "feature_name": "ecg_hrv_dominant_frequency", "orden_checkpoint": 114}, {"senal": "ECG", "codigo_tesis": "ECG_115", "feature_name": "ecg_hrv_mean_frequency", "orden_checkpoint": 115}, {"senal": "ECG", "codigo_tesis": "ECG_116", "feature_name": "ecg_hrv_median_frequency", "orden_checkpoint": 116}, {"senal": "ECG", "codigo_tesis": "ECG_117", "feature_name": "ecg_hrv_spectral_entropy", "orden_checkpoint": 117}, {"senal": "ECG", "codigo_tesis": "ECG_118", "feature_name": "ecg_hrv_total_spectral_power", "orden_checkpoint": 118}, {"senal": "ECG", "codigo_tesis": "ECG_119", "feature_name": "ecg_hrv_spectral_centroid", "orden_checkpoint": 119}, {"senal": "ECG", "codigo_tesis": "ECG_120", "feature_name": "ecg_hrv_spectral_spread", "orden_checkpoint": 120}, {"senal": "ECG", "codigo_tesis": "ECG_121", "feature_name": "ecg_hrv_spectral_skewness", "orden_checkpoint": 121}, {"senal": "ECG", "codigo_tesis": "ECG_122", "feature_name": "ecg_hrv_spectral_kurtosis", "orden_checkpoint": 122}, {"senal": "ECG", "codigo_tesis": "ECG_123", "feature_name": "ecg_hrv_power_vlf", "orden_checkpoint": 123}, {"senal": "ECG", "codigo_tesis": "ECG_124", "feature_name": "ecg_hrv_relative_power_vlf", "orden_checkpoint": 124}, {"senal": "ECG", "codigo_tesis": "ECG_125", "feature_name": "ecg_hrv_power_lf", "orden_checkpoint": 125}, {"senal": "ECG", "codigo_tesis": "ECG_126", "feature_name": "ecg_hrv_relative_power_lf", "orden_checkpoint": 126}, {"senal": "ECG", "codigo_tesis": "ECG_127", "feature_name": "ecg_hrv_power_hf", "orden_checkpoint": 127}, {"senal": "ECG", "codigo_tesis": "ECG_128", "feature_name": "ecg_hrv_relative_power_hf", "orden_checkpoint": 128}, {"senal": "ECG", "codigo_tesis": "ECG_129", "feature_name": "ecg_hrv_lf_hf_ratio", "orden_checkpoint": 129}, {"senal": "EDA", "codigo_tesis": "EDA_20", "feature_name": "eda_raw_mean", "orden_checkpoint": 1}, {"senal": "EDA", "codigo_tesis": "EDA_25", "feature_name": "eda_raw_median", "orden_checkpoint": 2}, {"senal": "EDA", "codigo_tesis": "EDA_61", "feature_name": "eda_raw_std", "orden_checkpoint": 3}, {"senal": "EDA", "codigo_tesis": "EDA_63", "feature_name": "eda_raw_variance", "orden_checkpoint": 4}, {"senal": "EDA", "codigo_tesis": "EDA_27", "feature_name": "eda_raw_min", "orden_checkpoint": 5}, {"senal": "EDA", "codigo_tesis": "EDA_19", "feature_name": "eda_raw_max", "orden_checkpoint": 6}, {"senal": "EDA", "codigo_tesis": "EDA_42", "feature_name": "eda_raw_range", "orden_checkpoint": 7}, {"senal": "EDA", "codigo_tesis": "EDA_34", "feature_name": "eda_raw_q01", "orden_checkpoint": 8}, {"senal": "EDA", "codigo_tesis": "EDA_35", "feature_name": "eda_raw_q05", "orden_checkpoint": 9}, {"senal": "EDA", "codigo_tesis": "EDA_36", "feature_name": "eda_raw_q10", "orden_checkpoint": 10}, {"senal": "EDA", "codigo_tesis": "EDA_37", "feature_name": "eda_raw_q25", "orden_checkpoint": 11}, {"senal": "EDA", "codigo_tesis": "EDA_38", "feature_name": "eda_raw_q75", "orden_checkpoint": 12}, {"senal": "EDA", "codigo_tesis": "EDA_39", "feature_name": "eda_raw_q90", "orden_checkpoint": 13}, {"senal": "EDA", "codigo_tesis": "EDA_40", "feature_name": "eda_raw_q95", "orden_checkpoint": 14}, {"senal": "EDA", "codigo_tesis": "EDA_41", "feature_name": "eda_raw_q99", "orden_checkpoint": 15}, {"senal": "EDA", "codigo_tesis": "EDA_13", "feature_name": "eda_raw_iqr", "orden_checkpoint": 16}, {"senal": "EDA", "codigo_tesis": "EDA_17", "feature_name": "eda_raw_mad_mean", "orden_checkpoint": 17}, {"senal": "EDA", "codigo_tesis": "EDA_18", "feature_name": "eda_raw_mad_median", "orden_checkpoint": 18}, {"senal": "EDA", "codigo_tesis": "EDA_47", "feature_name": "eda_raw_rms", "orden_checkpoint": 19}, {"senal": "EDA", "codigo_tesis": "EDA_21", "feature_name": "eda_raw_mean_absolute_value", "orden_checkpoint": 20}, {"senal": "EDA", "codigo_tesis": "EDA_10", "feature_name": "eda_raw_energy", "orden_checkpoint": 21}, {"senal": "EDA", "codigo_tesis": "EDA_24", "feature_name": "eda_raw_mean_power", "orden_checkpoint": 22}, {"senal": "EDA", "codigo_tesis": "EDA_2", "feature_name": "eda_raw_auc_absolute", "orden_checkpoint": 23}, {"senal": "EDA", "codigo_tesis": "EDA_3", "feature_name": "eda_raw_auc_signed", "orden_checkpoint": 24}, {"senal": "EDA", "codigo_tesis": "EDA_54", "feature_name": "eda_raw_skewness", "orden_checkpoint": 25}, {"senal": "EDA", "codigo_tesis": "EDA_15", "feature_name": "eda_raw_kurtosis", "orden_checkpoint": 26}, {"senal": "EDA", "codigo_tesis": "EDA_55", "feature_name": "eda_raw_slope", "orden_checkpoint": 27}, {"senal": "EDA", "codigo_tesis": "EDA_7", "feature_name": "eda_raw_delta", "orden_checkpoint": 28}, {"senal": "EDA", "codigo_tesis": "EDA_5", "feature_name": "eda_raw_coefficient_variation", "orden_checkpoint": 29}, {"senal": "EDA", "codigo_tesis": "EDA_90", "feature_name": "eda_raw_zero_crossing_rate", "orden_checkpoint": 30}, {"senal": "EDA", "codigo_tesis": "EDA_22", "feature_name": "eda_raw_mean_crossing_rate", "orden_checkpoint": 31}, {"senal": "EDA", "codigo_tesis": "EDA_64", "feature_name": "eda_raw_waveform_length", "orden_checkpoint": 32}, {"senal": "EDA", "codigo_tesis": "EDA_6", "feature_name": "eda_raw_crest_factor", "orden_checkpoint": 33}, {"senal": "EDA", "codigo_tesis": "EDA_53", "feature_name": "eda_raw_shape_factor", "orden_checkpoint": 34}, {"senal": "EDA", "codigo_tesis": "EDA_12", "feature_name": "eda_raw_impulse_factor", "orden_checkpoint": 35}, {"senal": "EDA", "codigo_tesis": "EDA_4", "feature_name": "eda_raw_clearance_factor", "orden_checkpoint": 36}, {"senal": "EDA", "codigo_tesis": "EDA_9", "feature_name": "eda_raw_dominant_frequency", "orden_checkpoint": 37}, {"senal": "EDA", "codigo_tesis": "EDA_23", "feature_name": "eda_raw_mean_frequency", "orden_checkpoint": 38}, {"senal": "EDA", "codigo_tesis": "EDA_26", "feature_name": "eda_raw_median_frequency", "orden_checkpoint": 39}, {"senal": "EDA", "codigo_tesis": "EDA_57", "feature_name": "eda_raw_spectral_entropy", "orden_checkpoint": 40}, {"senal": "EDA", "codigo_tesis": "EDA_62", "feature_name": "eda_raw_total_spectral_power", "orden_checkpoint": 41}, {"senal": "EDA", "codigo_tesis": "EDA_56", "feature_name": "eda_raw_spectral_centroid", "orden_checkpoint": 42}, {"senal": "EDA", "codigo_tesis": "EDA_60", "feature_name": "eda_raw_spectral_spread", "orden_checkpoint": 43}, {"senal": "EDA", "codigo_tesis": "EDA_59", "feature_name": "eda_raw_spectral_skewness", "orden_checkpoint": 44}, {"senal": "EDA", "codigo_tesis": "EDA_58", "feature_name": "eda_raw_spectral_kurtosis", "orden_checkpoint": 45}, {"senal": "EDA", "codigo_tesis": "EDA_32", "feature_name": "eda_raw_power_ultra_slow", "orden_checkpoint": 46}, {"senal": "EDA", "codigo_tesis": "EDA_45", "feature_name": "eda_raw_relative_power_ultra_slow", "orden_checkpoint": 47}, {"senal": "EDA", "codigo_tesis": "EDA_31", "feature_name": "eda_raw_power_slow", "orden_checkpoint": 48}, {"senal": "EDA", "codigo_tesis": "EDA_44", "feature_name": "eda_raw_relative_power_slow", "orden_checkpoint": 49}, {"senal": "EDA", "codigo_tesis": "EDA_30", "feature_name": "eda_raw_power_fast", "orden_checkpoint": 50}, {"senal": "EDA", "codigo_tesis": "EDA_43", "feature_name": "eda_raw_relative_power_fast", "orden_checkpoint": 51}, {"senal": "EDA", "codigo_tesis": "EDA_33", "feature_name": "eda_raw_power_very_fast", "orden_checkpoint": 52}, {"senal": "EDA", "codigo_tesis": "EDA_46", "feature_name": "eda_raw_relative_power_very_fast", "orden_checkpoint": 53}, {"senal": "EDA", "codigo_tesis": "EDA_52", "feature_name": "eda_raw_sample_entropy", "orden_checkpoint": 54}, {"senal": "EDA", "codigo_tesis": "EDA_1", "feature_name": "eda_raw_approximate_entropy", "orden_checkpoint": 55}, {"senal": "EDA", "codigo_tesis": "EDA_28", "feature_name": "eda_raw_permutation_entropy", "orden_checkpoint": 56}, {"senal": "EDA", "codigo_tesis": "EDA_16", "feature_name": "eda_raw_lempel_ziv_complexity", "orden_checkpoint": 57}, {"senal": "EDA", "codigo_tesis": "EDA_29", "feature_name": "eda_raw_petrosian_fd", "orden_checkpoint": 58}, {"senal": "EDA", "codigo_tesis": "EDA_14", "feature_name": "eda_raw_katz_fd", "orden_checkpoint": 59}, {"senal": "EDA", "codigo_tesis": "EDA_11", "feature_name": "eda_raw_higuchi_fd", "orden_checkpoint": 60}, {"senal": "EDA", "codigo_tesis": "EDA_8", "feature_name": "eda_raw_dfa", "orden_checkpoint": 61}, {"senal": "EDA", "codigo_tesis": "EDA_51", "feature_name": "eda_raw_rqa_recurrence_rate", "orden_checkpoint": 62}, {"senal": "EDA", "codigo_tesis": "EDA_48", "feature_name": "eda_raw_rqa_determinism", "orden_checkpoint": 63}, {"senal": "EDA", "codigo_tesis": "EDA_50", "feature_name": "eda_raw_rqa_laminarity", "orden_checkpoint": 64}, {"senal": "EDA", "codigo_tesis": "EDA_49", "feature_name": "eda_raw_rqa_diagonal_entropy", "orden_checkpoint": 65}, {"senal": "EDA", "codigo_tesis": "EDA_65", "feature_name": "eda_raw_wavelet_A_energy", "orden_checkpoint": 66}, {"senal": "EDA", "codigo_tesis": "EDA_68", "feature_name": "eda_raw_wavelet_A_relative_energy", "orden_checkpoint": 67}, {"senal": "EDA", "codigo_tesis": "EDA_67", "feature_name": "eda_raw_wavelet_A_mean_abs", "orden_checkpoint": 68}, {"senal": "EDA", "codigo_tesis": "EDA_69", "feature_name": "eda_raw_wavelet_A_std", "orden_checkpoint": 69}, {"senal": "EDA", "codigo_tesis": "EDA_66", "feature_name": "eda_raw_wavelet_A_entropy", "orden_checkpoint": 70}, {"senal": "EDA", "codigo_tesis": "EDA_85", "feature_name": "eda_raw_wavelet_D4_energy", "orden_checkpoint": 71}, {"senal": "EDA", "codigo_tesis": "EDA_88", "feature_name": "eda_raw_wavelet_D4_relative_energy", "orden_checkpoint": 72}, {"senal": "EDA", "codigo_tesis": "EDA_87", "feature_name": "eda_raw_wavelet_D4_mean_abs", "orden_checkpoint": 73}, {"senal": "EDA", "codigo_tesis": "EDA_89", "feature_name": "eda_raw_wavelet_D4_std", "orden_checkpoint": 74}, {"senal": "EDA", "codigo_tesis": "EDA_86", "feature_name": "eda_raw_wavelet_D4_entropy", "orden_checkpoint": 75}, {"senal": "EDA", "codigo_tesis": "EDA_80", "feature_name": "eda_raw_wavelet_D3_energy", "orden_checkpoint": 76}, {"senal": "EDA", "codigo_tesis": "EDA_83", "feature_name": "eda_raw_wavelet_D3_relative_energy", "orden_checkpoint": 77}, {"senal": "EDA", "codigo_tesis": "EDA_82", "feature_name": "eda_raw_wavelet_D3_mean_abs", "orden_checkpoint": 78}, {"senal": "EDA", "codigo_tesis": "EDA_84", "feature_name": "eda_raw_wavelet_D3_std", "orden_checkpoint": 79}, {"senal": "EDA", "codigo_tesis": "EDA_81", "feature_name": "eda_raw_wavelet_D3_entropy", "orden_checkpoint": 80}, {"senal": "EDA", "codigo_tesis": "EDA_75", "feature_name": "eda_raw_wavelet_D2_energy", "orden_checkpoint": 81}, {"senal": "EDA", "codigo_tesis": "EDA_78", "feature_name": "eda_raw_wavelet_D2_relative_energy", "orden_checkpoint": 82}, {"senal": "EDA", "codigo_tesis": "EDA_77", "feature_name": "eda_raw_wavelet_D2_mean_abs", "orden_checkpoint": 83}, {"senal": "EDA", "codigo_tesis": "EDA_79", "feature_name": "eda_raw_wavelet_D2_std", "orden_checkpoint": 84}, {"senal": "EDA", "codigo_tesis": "EDA_76", "feature_name": "eda_raw_wavelet_D2_entropy", "orden_checkpoint": 85}, {"senal": "EDA", "codigo_tesis": "EDA_70", "feature_name": "eda_raw_wavelet_D1_energy", "orden_checkpoint": 86}, {"senal": "EDA", "codigo_tesis": "EDA_73", "feature_name": "eda_raw_wavelet_D1_relative_energy", "orden_checkpoint": 87}, {"senal": "EDA", "codigo_tesis": "EDA_72", "feature_name": "eda_raw_wavelet_D1_mean_abs", "orden_checkpoint": 88}, {"senal": "EDA", "codigo_tesis": "EDA_74", "feature_name": "eda_raw_wavelet_D1_std", "orden_checkpoint": 89}, {"senal": "EDA", "codigo_tesis": "EDA_71", "feature_name": "eda_raw_wavelet_D1_entropy", "orden_checkpoint": 90}, {"senal": "EDA", "codigo_tesis": "EDA_91", "feature_name": "eda_tonic_mean", "orden_checkpoint": 91}, {"senal": "EDA", "codigo_tesis": "EDA_92", "feature_name": "eda_tonic_median", "orden_checkpoint": 92}, {"senal": "EDA", "codigo_tesis": "EDA_93", "feature_name": "eda_tonic_std", "orden_checkpoint": 93}, {"senal": "EDA", "codigo_tesis": "EDA_94", "feature_name": "eda_tonic_variance", "orden_checkpoint": 94}, {"senal": "EDA", "codigo_tesis": "EDA_95", "feature_name": "eda_tonic_min", "orden_checkpoint": 95}, {"senal": "EDA", "codigo_tesis": "EDA_96", "feature_name": "eda_tonic_max", "orden_checkpoint": 96}, {"senal": "EDA", "codigo_tesis": "EDA_97", "feature_name": "eda_tonic_range", "orden_checkpoint": 97}, {"senal": "EDA", "codigo_tesis": "EDA_98", "feature_name": "eda_tonic_q01", "orden_checkpoint": 98}, {"senal": "EDA", "codigo_tesis": "EDA_99", "feature_name": "eda_tonic_q05", "orden_checkpoint": 99}, {"senal": "EDA", "codigo_tesis": "EDA_100", "feature_name": "eda_tonic_q10", "orden_checkpoint": 100}, {"senal": "EDA", "codigo_tesis": "EDA_101", "feature_name": "eda_tonic_q25", "orden_checkpoint": 101}, {"senal": "EDA", "codigo_tesis": "EDA_102", "feature_name": "eda_tonic_q75", "orden_checkpoint": 102}, {"senal": "EDA", "codigo_tesis": "EDA_103", "feature_name": "eda_tonic_q90", "orden_checkpoint": 103}, {"senal": "EDA", "codigo_tesis": "EDA_104", "feature_name": "eda_tonic_q95", "orden_checkpoint": 104}, {"senal": "EDA", "codigo_tesis": "EDA_105", "feature_name": "eda_tonic_q99", "orden_checkpoint": 105}, {"senal": "EDA", "codigo_tesis": "EDA_106", "feature_name": "eda_tonic_iqr", "orden_checkpoint": 106}, {"senal": "EDA", "codigo_tesis": "EDA_107", "feature_name": "eda_tonic_mad_mean", "orden_checkpoint": 107}, {"senal": "EDA", "codigo_tesis": "EDA_108", "feature_name": "eda_tonic_mad_median", "orden_checkpoint": 108}, {"senal": "EDA", "codigo_tesis": "EDA_109", "feature_name": "eda_tonic_rms", "orden_checkpoint": 109}, {"senal": "EDA", "codigo_tesis": "EDA_110", "feature_name": "eda_tonic_mean_absolute_value", "orden_checkpoint": 110}, {"senal": "EDA", "codigo_tesis": "EDA_111", "feature_name": "eda_tonic_energy", "orden_checkpoint": 111}, {"senal": "EDA", "codigo_tesis": "EDA_112", "feature_name": "eda_tonic_mean_power", "orden_checkpoint": 112}, {"senal": "EDA", "codigo_tesis": "EDA_113", "feature_name": "eda_tonic_auc_absolute", "orden_checkpoint": 113}, {"senal": "EDA", "codigo_tesis": "EDA_114", "feature_name": "eda_tonic_auc_signed", "orden_checkpoint": 114}, {"senal": "EDA", "codigo_tesis": "EDA_115", "feature_name": "eda_tonic_skewness", "orden_checkpoint": 115}, {"senal": "EDA", "codigo_tesis": "EDA_116", "feature_name": "eda_tonic_kurtosis", "orden_checkpoint": 116}, {"senal": "EDA", "codigo_tesis": "EDA_117", "feature_name": "eda_tonic_slope", "orden_checkpoint": 117}, {"senal": "EDA", "codigo_tesis": "EDA_118", "feature_name": "eda_tonic_delta", "orden_checkpoint": 118}, {"senal": "EDA", "codigo_tesis": "EDA_119", "feature_name": "eda_tonic_coefficient_variation", "orden_checkpoint": 119}, {"senal": "EDA", "codigo_tesis": "EDA_120", "feature_name": "eda_tonic_zero_crossing_rate", "orden_checkpoint": 120}, {"senal": "EDA", "codigo_tesis": "EDA_121", "feature_name": "eda_tonic_mean_crossing_rate", "orden_checkpoint": 121}, {"senal": "EDA", "codigo_tesis": "EDA_122", "feature_name": "eda_tonic_waveform_length", "orden_checkpoint": 122}, {"senal": "EDA", "codigo_tesis": "EDA_123", "feature_name": "eda_tonic_crest_factor", "orden_checkpoint": 123}, {"senal": "EDA", "codigo_tesis": "EDA_124", "feature_name": "eda_tonic_shape_factor", "orden_checkpoint": 124}, {"senal": "EDA", "codigo_tesis": "EDA_125", "feature_name": "eda_tonic_impulse_factor", "orden_checkpoint": 125}, {"senal": "EDA", "codigo_tesis": "EDA_126", "feature_name": "eda_tonic_clearance_factor", "orden_checkpoint": 126}, {"senal": "EDA", "codigo_tesis": "EDA_127", "feature_name": "eda_phasic_mean", "orden_checkpoint": 127}, {"senal": "EDA", "codigo_tesis": "EDA_128", "feature_name": "eda_phasic_median", "orden_checkpoint": 128}, {"senal": "EDA", "codigo_tesis": "EDA_129", "feature_name": "eda_phasic_std", "orden_checkpoint": 129}, {"senal": "EDA", "codigo_tesis": "EDA_130", "feature_name": "eda_phasic_variance", "orden_checkpoint": 130}, {"senal": "EDA", "codigo_tesis": "EDA_131", "feature_name": "eda_phasic_min", "orden_checkpoint": 131}, {"senal": "EDA", "codigo_tesis": "EDA_132", "feature_name": "eda_phasic_max", "orden_checkpoint": 132}, {"senal": "EDA", "codigo_tesis": "EDA_133", "feature_name": "eda_phasic_range", "orden_checkpoint": 133}, {"senal": "EDA", "codigo_tesis": "EDA_134", "feature_name": "eda_phasic_q01", "orden_checkpoint": 134}, {"senal": "EDA", "codigo_tesis": "EDA_135", "feature_name": "eda_phasic_q05", "orden_checkpoint": 135}, {"senal": "EDA", "codigo_tesis": "EDA_136", "feature_name": "eda_phasic_q10", "orden_checkpoint": 136}, {"senal": "EDA", "codigo_tesis": "EDA_137", "feature_name": "eda_phasic_q25", "orden_checkpoint": 137}, {"senal": "EDA", "codigo_tesis": "EDA_138", "feature_name": "eda_phasic_q75", "orden_checkpoint": 138}, {"senal": "EDA", "codigo_tesis": "EDA_139", "feature_name": "eda_phasic_q90", "orden_checkpoint": 139}, {"senal": "EDA", "codigo_tesis": "EDA_140", "feature_name": "eda_phasic_q95", "orden_checkpoint": 140}, {"senal": "EDA", "codigo_tesis": "EDA_141", "feature_name": "eda_phasic_q99", "orden_checkpoint": 141}, {"senal": "EDA", "codigo_tesis": "EDA_142", "feature_name": "eda_phasic_iqr", "orden_checkpoint": 142}, {"senal": "EDA", "codigo_tesis": "EDA_143", "feature_name": "eda_phasic_mad_mean", "orden_checkpoint": 143}, {"senal": "EDA", "codigo_tesis": "EDA_144", "feature_name": "eda_phasic_mad_median", "orden_checkpoint": 144}, {"senal": "EDA", "codigo_tesis": "EDA_145", "feature_name": "eda_phasic_rms", "orden_checkpoint": 145}, {"senal": "EDA", "codigo_tesis": "EDA_146", "feature_name": "eda_phasic_mean_absolute_value", "orden_checkpoint": 146}, {"senal": "EDA", "codigo_tesis": "EDA_147", "feature_name": "eda_phasic_energy", "orden_checkpoint": 147}, {"senal": "EDA", "codigo_tesis": "EDA_148", "feature_name": "eda_phasic_mean_power", "orden_checkpoint": 148}, {"senal": "EDA", "codigo_tesis": "EDA_149", "feature_name": "eda_phasic_auc_absolute", "orden_checkpoint": 149}, {"senal": "EDA", "codigo_tesis": "EDA_150", "feature_name": "eda_phasic_auc_signed", "orden_checkpoint": 150}, {"senal": "EDA", "codigo_tesis": "EDA_151", "feature_name": "eda_phasic_skewness", "orden_checkpoint": 151}, {"senal": "EDA", "codigo_tesis": "EDA_152", "feature_name": "eda_phasic_kurtosis", "orden_checkpoint": 152}, {"senal": "EDA", "codigo_tesis": "EDA_153", "feature_name": "eda_phasic_slope", "orden_checkpoint": 153}, {"senal": "EDA", "codigo_tesis": "EDA_154", "feature_name": "eda_phasic_delta", "orden_checkpoint": 154}, {"senal": "EDA", "codigo_tesis": "EDA_155", "feature_name": "eda_phasic_coefficient_variation", "orden_checkpoint": 155}, {"senal": "EDA", "codigo_tesis": "EDA_156", "feature_name": "eda_phasic_zero_crossing_rate", "orden_checkpoint": 156}, {"senal": "EDA", "codigo_tesis": "EDA_157", "feature_name": "eda_phasic_mean_crossing_rate", "orden_checkpoint": 157}, {"senal": "EDA", "codigo_tesis": "EDA_158", "feature_name": "eda_phasic_waveform_length", "orden_checkpoint": 158}, {"senal": "EDA", "codigo_tesis": "EDA_159", "feature_name": "eda_phasic_crest_factor", "orden_checkpoint": 159}, {"senal": "EDA", "codigo_tesis": "EDA_160", "feature_name": "eda_phasic_shape_factor", "orden_checkpoint": 160}, {"senal": "EDA", "codigo_tesis": "EDA_161", "feature_name": "eda_phasic_impulse_factor", "orden_checkpoint": 161}, {"senal": "EDA", "codigo_tesis": "EDA_162", "feature_name": "eda_phasic_clearance_factor", "orden_checkpoint": 162}, {"senal": "EDA", "codigo_tesis": "EDA_163", "feature_name": "eda_tonic_sample_entropy", "orden_checkpoint": 163}, {"senal": "EDA", "codigo_tesis": "EDA_164", "feature_name": "eda_tonic_approximate_entropy", "orden_checkpoint": 164}, {"senal": "EDA", "codigo_tesis": "EDA_165", "feature_name": "eda_tonic_permutation_entropy", "orden_checkpoint": 165}, {"senal": "EDA", "codigo_tesis": "EDA_166", "feature_name": "eda_tonic_lempel_ziv_complexity", "orden_checkpoint": 166}, {"senal": "EDA", "codigo_tesis": "EDA_167", "feature_name": "eda_tonic_petrosian_fd", "orden_checkpoint": 167}, {"senal": "EDA", "codigo_tesis": "EDA_168", "feature_name": "eda_tonic_katz_fd", "orden_checkpoint": 168}, {"senal": "EDA", "codigo_tesis": "EDA_169", "feature_name": "eda_tonic_higuchi_fd", "orden_checkpoint": 169}, {"senal": "EDA", "codigo_tesis": "EDA_170", "feature_name": "eda_tonic_dfa", "orden_checkpoint": 170}, {"senal": "EDA", "codigo_tesis": "EDA_171", "feature_name": "eda_tonic_rqa_recurrence_rate", "orden_checkpoint": 171}, {"senal": "EDA", "codigo_tesis": "EDA_172", "feature_name": "eda_tonic_rqa_determinism", "orden_checkpoint": 172}, {"senal": "EDA", "codigo_tesis": "EDA_173", "feature_name": "eda_tonic_rqa_laminarity", "orden_checkpoint": 173}, {"senal": "EDA", "codigo_tesis": "EDA_174", "feature_name": "eda_tonic_rqa_diagonal_entropy", "orden_checkpoint": 174}, {"senal": "EDA", "codigo_tesis": "EDA_175", "feature_name": "eda_phasic_sample_entropy", "orden_checkpoint": 175}, {"senal": "EDA", "codigo_tesis": "EDA_176", "feature_name": "eda_phasic_approximate_entropy", "orden_checkpoint": 176}, {"senal": "EDA", "codigo_tesis": "EDA_177", "feature_name": "eda_phasic_permutation_entropy", "orden_checkpoint": 177}, {"senal": "EDA", "codigo_tesis": "EDA_178", "feature_name": "eda_phasic_lempel_ziv_complexity", "orden_checkpoint": 178}, {"senal": "EDA", "codigo_tesis": "EDA_179", "feature_name": "eda_phasic_petrosian_fd", "orden_checkpoint": 179}, {"senal": "EDA", "codigo_tesis": "EDA_180", "feature_name": "eda_phasic_katz_fd", "orden_checkpoint": 180}, {"senal": "EDA", "codigo_tesis": "EDA_181", "feature_name": "eda_phasic_higuchi_fd", "orden_checkpoint": 181}, {"senal": "EDA", "codigo_tesis": "EDA_182", "feature_name": "eda_phasic_dfa", "orden_checkpoint": 182}, {"senal": "EDA", "codigo_tesis": "EDA_183", "feature_name": "eda_phasic_rqa_recurrence_rate", "orden_checkpoint": 183}, {"senal": "EDA", "codigo_tesis": "EDA_184", "feature_name": "eda_phasic_rqa_determinism", "orden_checkpoint": 184}, {"senal": "EDA", "codigo_tesis": "EDA_185", "feature_name": "eda_phasic_rqa_laminarity", "orden_checkpoint": 185}, {"senal": "EDA", "codigo_tesis": "EDA_186", "feature_name": "eda_phasic_rqa_diagonal_entropy", "orden_checkpoint": 186}, {"senal": "EDA", "codigo_tesis": "EDA_187", "feature_name": "eda_phasic_wavelet_A_energy", "orden_checkpoint": 187}, {"senal": "EDA", "codigo_tesis": "EDA_188", "feature_name": "eda_phasic_wavelet_A_relative_energy", "orden_checkpoint": 188}, {"senal": "EDA", "codigo_tesis": "EDA_189", "feature_name": "eda_phasic_wavelet_A_mean_abs", "orden_checkpoint": 189}, {"senal": "EDA", "codigo_tesis": "EDA_190", "feature_name": "eda_phasic_wavelet_A_std", "orden_checkpoint": 190}, {"senal": "EDA", "codigo_tesis": "EDA_191", "feature_name": "eda_phasic_wavelet_A_entropy", "orden_checkpoint": 191}, {"senal": "EDA", "codigo_tesis": "EDA_192", "feature_name": "eda_phasic_wavelet_D4_energy", "orden_checkpoint": 192}, {"senal": "EDA", "codigo_tesis": "EDA_193", "feature_name": "eda_phasic_wavelet_D4_relative_energy", "orden_checkpoint": 193}, {"senal": "EDA", "codigo_tesis": "EDA_194", "feature_name": "eda_phasic_wavelet_D4_mean_abs", "orden_checkpoint": 194}, {"senal": "EDA", "codigo_tesis": "EDA_195", "feature_name": "eda_phasic_wavelet_D4_std", "orden_checkpoint": 195}, {"senal": "EDA", "codigo_tesis": "EDA_196", "feature_name": "eda_phasic_wavelet_D4_entropy", "orden_checkpoint": 196}, {"senal": "EDA", "codigo_tesis": "EDA_197", "feature_name": "eda_phasic_wavelet_D3_energy", "orden_checkpoint": 197}, {"senal": "EDA", "codigo_tesis": "EDA_198", "feature_name": "eda_phasic_wavelet_D3_relative_energy", "orden_checkpoint": 198}, {"senal": "EDA", "codigo_tesis": "EDA_199", "feature_name": "eda_phasic_wavelet_D3_mean_abs", "orden_checkpoint": 199}, {"senal": "EDA", "codigo_tesis": "EDA_200", "feature_name": "eda_phasic_wavelet_D3_std", "orden_checkpoint": 200}, {"senal": "EDA", "codigo_tesis": "EDA_201", "feature_name": "eda_phasic_wavelet_D3_entropy", "orden_checkpoint": 201}, {"senal": "EDA", "codigo_tesis": "EDA_202", "feature_name": "eda_phasic_wavelet_D2_energy", "orden_checkpoint": 202}, {"senal": "EDA", "codigo_tesis": "EDA_203", "feature_name": "eda_phasic_wavelet_D2_relative_energy", "orden_checkpoint": 203}, {"senal": "EDA", "codigo_tesis": "EDA_204", "feature_name": "eda_phasic_wavelet_D2_mean_abs", "orden_checkpoint": 204}, {"senal": "EDA", "codigo_tesis": "EDA_205", "feature_name": "eda_phasic_wavelet_D2_std", "orden_checkpoint": 205}, {"senal": "EDA", "codigo_tesis": "EDA_206", "feature_name": "eda_phasic_wavelet_D2_entropy", "orden_checkpoint": 206}, {"senal": "EDA", "codigo_tesis": "EDA_207", "feature_name": "eda_phasic_wavelet_D1_energy", "orden_checkpoint": 207}, {"senal": "EDA", "codigo_tesis": "EDA_208", "feature_name": "eda_phasic_wavelet_D1_relative_energy", "orden_checkpoint": 208}, {"senal": "EDA", "codigo_tesis": "EDA_209", "feature_name": "eda_phasic_wavelet_D1_mean_abs", "orden_checkpoint": 209}, {"senal": "EDA", "codigo_tesis": "EDA_210", "feature_name": "eda_phasic_wavelet_D1_std", "orden_checkpoint": 210}, {"senal": "EDA", "codigo_tesis": "EDA_211", "feature_name": "eda_phasic_wavelet_D1_entropy", "orden_checkpoint": 211}, {"senal": "EDA", "codigo_tesis": "EDA_212", "feature_name": "eda_scr_count", "orden_checkpoint": 212}, {"senal": "EDA", "codigo_tesis": "EDA_213", "feature_name": "eda_scr_rate_per_minute", "orden_checkpoint": 213}, {"senal": "EDA", "codigo_tesis": "EDA_214", "feature_name": "eda_scr_mean_amplitude", "orden_checkpoint": 214}, {"senal": "EDA", "codigo_tesis": "EDA_215", "feature_name": "eda_scr_median_amplitude", "orden_checkpoint": 215}, {"senal": "EDA", "codigo_tesis": "EDA_216", "feature_name": "eda_scr_std_amplitude", "orden_checkpoint": 216}, {"senal": "EDA", "codigo_tesis": "EDA_217", "feature_name": "eda_scr_max_amplitude", "orden_checkpoint": 217}, {"senal": "EDA", "codigo_tesis": "EDA_218", "feature_name": "eda_scr_sum_amplitude", "orden_checkpoint": 218}, {"senal": "EDA", "codigo_tesis": "EDA_219", "feature_name": "eda_scr_mean_rise_time", "orden_checkpoint": 219}, {"senal": "EDA", "codigo_tesis": "EDA_220", "feature_name": "eda_scr_std_rise_time", "orden_checkpoint": 220}, {"senal": "EDA", "codigo_tesis": "EDA_221", "feature_name": "eda_scr_mean_recovery_time", "orden_checkpoint": 221}, {"senal": "EDA", "codigo_tesis": "EDA_222", "feature_name": "eda_scr_std_recovery_time", "orden_checkpoint": 222}, {"senal": "EDA", "codigo_tesis": "EDA_223", "feature_name": "eda_phasic_auc", "orden_checkpoint": 223}, {"senal": "EDA", "codigo_tesis": "EDA_224", "feature_name": "eda_tonic_phasic_ratio", "orden_checkpoint": 224}, {"senal": "EDA", "codigo_tesis": "EDA_225", "feature_name": "eda_scr_mean_onset_to_peak", "orden_checkpoint": 225}, {"senal": "EDA", "codigo_tesis": "EDA_226", "feature_name": "eda_scr_std_onset_to_peak", "orden_checkpoint": 226}, {"senal": "RESP", "codigo_tesis": "RESP_30", "feature_name": "resp_mean", "orden_checkpoint": 1}, {"senal": "RESP", "codigo_tesis": "RESP_35", "feature_name": "resp_median", "orden_checkpoint": 2}, {"senal": "RESP", "codigo_tesis": "RESP_71", "feature_name": "resp_std", "orden_checkpoint": 3}, {"senal": "RESP", "codigo_tesis": "RESP_73", "feature_name": "resp_variance", "orden_checkpoint": 4}, {"senal": "RESP", "codigo_tesis": "RESP_37", "feature_name": "resp_min", "orden_checkpoint": 5}, {"senal": "RESP", "codigo_tesis": "RESP_29", "feature_name": "resp_max", "orden_checkpoint": 6}, {"senal": "RESP", "codigo_tesis": "RESP_52", "feature_name": "resp_range", "orden_checkpoint": 7}, {"senal": "RESP", "codigo_tesis": "RESP_44", "feature_name": "resp_q01", "orden_checkpoint": 8}, {"senal": "RESP", "codigo_tesis": "RESP_45", "feature_name": "resp_q05", "orden_checkpoint": 9}, {"senal": "RESP", "codigo_tesis": "RESP_46", "feature_name": "resp_q10", "orden_checkpoint": 10}, {"senal": "RESP", "codigo_tesis": "RESP_47", "feature_name": "resp_q25", "orden_checkpoint": 11}, {"senal": "RESP", "codigo_tesis": "RESP_48", "feature_name": "resp_q75", "orden_checkpoint": 12}, {"senal": "RESP", "codigo_tesis": "RESP_49", "feature_name": "resp_q90", "orden_checkpoint": 13}, {"senal": "RESP", "codigo_tesis": "RESP_50", "feature_name": "resp_q95", "orden_checkpoint": 14}, {"senal": "RESP", "codigo_tesis": "RESP_51", "feature_name": "resp_q99", "orden_checkpoint": 15}, {"senal": "RESP", "codigo_tesis": "RESP_23", "feature_name": "resp_iqr", "orden_checkpoint": 16}, {"senal": "RESP", "codigo_tesis": "RESP_27", "feature_name": "resp_mad_mean", "orden_checkpoint": 17}, {"senal": "RESP", "codigo_tesis": "RESP_28", "feature_name": "resp_mad_median", "orden_checkpoint": 18}, {"senal": "RESP", "codigo_tesis": "RESP_57", "feature_name": "resp_rms", "orden_checkpoint": 19}, {"senal": "RESP", "codigo_tesis": "RESP_31", "feature_name": "resp_mean_absolute_value", "orden_checkpoint": 20}, {"senal": "RESP", "codigo_tesis": "RESP_15", "feature_name": "resp_energy", "orden_checkpoint": 21}, {"senal": "RESP", "codigo_tesis": "RESP_34", "feature_name": "resp_mean_power", "orden_checkpoint": 22}, {"senal": "RESP", "codigo_tesis": "RESP_2", "feature_name": "resp_auc_absolute", "orden_checkpoint": 23}, {"senal": "RESP", "codigo_tesis": "RESP_3", "feature_name": "resp_auc_signed", "orden_checkpoint": 24}, {"senal": "RESP", "codigo_tesis": "RESP_64", "feature_name": "resp_skewness", "orden_checkpoint": 25}, {"senal": "RESP", "codigo_tesis": "RESP_25", "feature_name": "resp_kurtosis", "orden_checkpoint": 26}, {"senal": "RESP", "codigo_tesis": "RESP_65", "feature_name": "resp_slope", "orden_checkpoint": 27}, {"senal": "RESP", "codigo_tesis": "RESP_12", "feature_name": "resp_delta", "orden_checkpoint": 28}, {"senal": "RESP", "codigo_tesis": "RESP_6", "feature_name": "resp_coefficient_variation", "orden_checkpoint": 29}, {"senal": "RESP", "codigo_tesis": "RESP_100", "feature_name": "resp_zero_crossing_rate", "orden_checkpoint": 30}, {"senal": "RESP", "codigo_tesis": "RESP_32", "feature_name": "resp_mean_crossing_rate", "orden_checkpoint": 31}, {"senal": "RESP", "codigo_tesis": "RESP_74", "feature_name": "resp_waveform_length", "orden_checkpoint": 32}, {"senal": "RESP", "codigo_tesis": "RESP_7", "feature_name": "resp_crest_factor", "orden_checkpoint": 33}, {"senal": "RESP", "codigo_tesis": "RESP_63", "feature_name": "resp_shape_factor", "orden_checkpoint": 34}, {"senal": "RESP", "codigo_tesis": "RESP_19", "feature_name": "resp_impulse_factor", "orden_checkpoint": 35}, {"senal": "RESP", "codigo_tesis": "RESP_5", "feature_name": "resp_clearance_factor", "orden_checkpoint": 36}, {"senal": "RESP", "codigo_tesis": "RESP_14", "feature_name": "resp_dominant_frequency", "orden_checkpoint": 37}, {"senal": "RESP", "codigo_tesis": "RESP_33", "feature_name": "resp_mean_frequency", "orden_checkpoint": 38}, {"senal": "RESP", "codigo_tesis": "RESP_36", "feature_name": "resp_median_frequency", "orden_checkpoint": 39}, {"senal": "RESP", "codigo_tesis": "RESP_67", "feature_name": "resp_spectral_entropy", "orden_checkpoint": 40}, {"senal": "RESP", "codigo_tesis": "RESP_72", "feature_name": "resp_total_spectral_power", "orden_checkpoint": 41}, {"senal": "RESP", "codigo_tesis": "RESP_66", "feature_name": "resp_spectral_centroid", "orden_checkpoint": 42}, {"senal": "RESP", "codigo_tesis": "RESP_70", "feature_name": "resp_spectral_spread", "orden_checkpoint": 43}, {"senal": "RESP", "codigo_tesis": "RESP_69", "feature_name": "resp_spectral_skewness", "orden_checkpoint": 44}, {"senal": "RESP", "codigo_tesis": "RESP_68", "feature_name": "resp_spectral_kurtosis", "orden_checkpoint": 45}, {"senal": "RESP", "codigo_tesis": "RESP_43", "feature_name": "resp_power_very_slow", "orden_checkpoint": 46}, {"senal": "RESP", "codigo_tesis": "RESP_56", "feature_name": "resp_relative_power_very_slow", "orden_checkpoint": 47}, {"senal": "RESP", "codigo_tesis": "RESP_42", "feature_name": "resp_power_normal_breathing", "orden_checkpoint": 48}, {"senal": "RESP", "codigo_tesis": "RESP_55", "feature_name": "resp_relative_power_normal_breathing", "orden_checkpoint": 49}, {"senal": "RESP", "codigo_tesis": "RESP_41", "feature_name": "resp_power_fast_breathing", "orden_checkpoint": 50}, {"senal": "RESP", "codigo_tesis": "RESP_54", "feature_name": "resp_relative_power_fast_breathing", "orden_checkpoint": 51}, {"senal": "RESP", "codigo_tesis": "RESP_62", "feature_name": "resp_sample_entropy", "orden_checkpoint": 52}, {"senal": "RESP", "codigo_tesis": "RESP_1", "feature_name": "resp_approximate_entropy", "orden_checkpoint": 53}, {"senal": "RESP", "codigo_tesis": "RESP_39", "feature_name": "resp_permutation_entropy", "orden_checkpoint": 54}, {"senal": "RESP", "codigo_tesis": "RESP_26", "feature_name": "resp_lempel_ziv_complexity", "orden_checkpoint": 55}, {"senal": "RESP", "codigo_tesis": "RESP_40", "feature_name": "resp_petrosian_fd", "orden_checkpoint": 56}, {"senal": "RESP", "codigo_tesis": "RESP_24", "feature_name": "resp_katz_fd", "orden_checkpoint": 57}, {"senal": "RESP", "codigo_tesis": "RESP_18", "feature_name": "resp_higuchi_fd", "orden_checkpoint": 58}, {"senal": "RESP", "codigo_tesis": "RESP_13", "feature_name": "resp_dfa", "orden_checkpoint": 59}, {"senal": "RESP", "codigo_tesis": "RESP_61", "feature_name": "resp_rqa_recurrence_rate", "orden_checkpoint": 60}, {"senal": "RESP", "codigo_tesis": "RESP_58", "feature_name": "resp_rqa_determinism", "orden_checkpoint": 61}, {"senal": "RESP", "codigo_tesis": "RESP_60", "feature_name": "resp_rqa_laminarity", "orden_checkpoint": 62}, {"senal": "RESP", "codigo_tesis": "RESP_59", "feature_name": "resp_rqa_diagonal_entropy", "orden_checkpoint": 63}, {"senal": "RESP", "codigo_tesis": "RESP_75", "feature_name": "resp_wavelet_A_energy", "orden_checkpoint": 64}, {"senal": "RESP", "codigo_tesis": "RESP_78", "feature_name": "resp_wavelet_A_relative_energy", "orden_checkpoint": 65}, {"senal": "RESP", "codigo_tesis": "RESP_77", "feature_name": "resp_wavelet_A_mean_abs", "orden_checkpoint": 66}, {"senal": "RESP", "codigo_tesis": "RESP_79", "feature_name": "resp_wavelet_A_std", "orden_checkpoint": 67}, {"senal": "RESP", "codigo_tesis": "RESP_76", "feature_name": "resp_wavelet_A_entropy", "orden_checkpoint": 68}, {"senal": "RESP", "codigo_tesis": "RESP_95", "feature_name": "resp_wavelet_D4_energy", "orden_checkpoint": 69}, {"senal": "RESP", "codigo_tesis": "RESP_98", "feature_name": "resp_wavelet_D4_relative_energy", "orden_checkpoint": 70}, {"senal": "RESP", "codigo_tesis": "RESP_97", "feature_name": "resp_wavelet_D4_mean_abs", "orden_checkpoint": 71}, {"senal": "RESP", "codigo_tesis": "RESP_99", "feature_name": "resp_wavelet_D4_std", "orden_checkpoint": 72}, {"senal": "RESP", "codigo_tesis": "RESP_96", "feature_name": "resp_wavelet_D4_entropy", "orden_checkpoint": 73}, {"senal": "RESP", "codigo_tesis": "RESP_90", "feature_name": "resp_wavelet_D3_energy", "orden_checkpoint": 74}, {"senal": "RESP", "codigo_tesis": "RESP_93", "feature_name": "resp_wavelet_D3_relative_energy", "orden_checkpoint": 75}, {"senal": "RESP", "codigo_tesis": "RESP_92", "feature_name": "resp_wavelet_D3_mean_abs", "orden_checkpoint": 76}, {"senal": "RESP", "codigo_tesis": "RESP_94", "feature_name": "resp_wavelet_D3_std", "orden_checkpoint": 77}, {"senal": "RESP", "codigo_tesis": "RESP_91", "feature_name": "resp_wavelet_D3_entropy", "orden_checkpoint": 78}, {"senal": "RESP", "codigo_tesis": "RESP_85", "feature_name": "resp_wavelet_D2_energy", "orden_checkpoint": 79}, {"senal": "RESP", "codigo_tesis": "RESP_88", "feature_name": "resp_wavelet_D2_relative_energy", "orden_checkpoint": 80}, {"senal": "RESP", "codigo_tesis": "RESP_87", "feature_name": "resp_wavelet_D2_mean_abs", "orden_checkpoint": 81}, {"senal": "RESP", "codigo_tesis": "RESP_89", "feature_name": "resp_wavelet_D2_std", "orden_checkpoint": 82}, {"senal": "RESP", "codigo_tesis": "RESP_86", "feature_name": "resp_wavelet_D2_entropy", "orden_checkpoint": 83}, {"senal": "RESP", "codigo_tesis": "RESP_80", "feature_name": "resp_wavelet_D1_energy", "orden_checkpoint": 84}, {"senal": "RESP", "codigo_tesis": "RESP_83", "feature_name": "resp_wavelet_D1_relative_energy", "orden_checkpoint": 85}, {"senal": "RESP", "codigo_tesis": "RESP_82", "feature_name": "resp_wavelet_D1_mean_abs", "orden_checkpoint": 86}, {"senal": "RESP", "codigo_tesis": "RESP_84", "feature_name": "resp_wavelet_D1_std", "orden_checkpoint": 87}, {"senal": "RESP", "codigo_tesis": "RESP_81", "feature_name": "resp_wavelet_D1_entropy", "orden_checkpoint": 88}, {"senal": "RESP", "codigo_tesis": "RESP_38", "feature_name": "resp_peak_count", "orden_checkpoint": 89}, {"senal": "RESP", "codigo_tesis": "RESP_53", "feature_name": "resp_rate_per_minute", "orden_checkpoint": 90}, {"senal": "RESP", "codigo_tesis": "RESP_9", "feature_name": "resp_cycle_mean_duration", "orden_checkpoint": 91}, {"senal": "RESP", "codigo_tesis": "RESP_10", "feature_name": "resp_cycle_median_duration", "orden_checkpoint": 92}, {"senal": "RESP", "codigo_tesis": "RESP_11", "feature_name": "resp_cycle_std_duration", "orden_checkpoint": 93}, {"senal": "RESP", "codigo_tesis": "RESP_8", "feature_name": "resp_cycle_cv", "orden_checkpoint": 94}, {"senal": "RESP", "codigo_tesis": "RESP_4", "feature_name": "resp_breath_to_breath_rmssd", "orden_checkpoint": 95}, {"senal": "RESP", "codigo_tesis": "RESP_21", "feature_name": "resp_inspiration_mean_duration", "orden_checkpoint": 96}, {"senal": "RESP", "codigo_tesis": "RESP_22", "feature_name": "resp_inspiration_std_duration", "orden_checkpoint": 97}, {"senal": "RESP", "codigo_tesis": "RESP_16", "feature_name": "resp_expiration_mean_duration", "orden_checkpoint": 98}, {"senal": "RESP", "codigo_tesis": "RESP_17", "feature_name": "resp_expiration_std_duration", "orden_checkpoint": 99}, {"senal": "RESP", "codigo_tesis": "RESP_20", "feature_name": "resp_inspiration_expiration_ratio", "orden_checkpoint": 100}]'''))
assert len(CATALOGO)==455
assert CATALOGO['feature_name'].is_unique and CATALOGO['codigo_tesis'].is_unique
assert CATALOGO.groupby('senal').size().to_dict()=={'ECG':129,'EDA':226,'RESP':100}
NAME_TO_CODE = dict(zip(CATALOGO.feature_name,CATALOGO.codigo_tesis))
FEATURE_NAMES = CATALOGO.feature_name.tolist()
FEATURE_SET = set(FEATURE_NAMES)
CATALOGO.to_csv(OUT_DIR/'catalogo_codigos_tesis.csv',index=False,encoding='utf-8-sig')
display(CATALOGO.groupby('senal').agg(cantidad=('feature_name','size')).reset_index())
display(CATALOGO.head(4))

,senal,cantidad
0,ECG,129
1,EDA,226
2,RESP,100


,senal,codigo_tesis,feature_name,orden_checkpoint
0,ECG,ECG_15,ecg_raw_mean,1
1,ECG,ECG_20,ecg_raw_median,2
2,ECG,ECG_49,ecg_raw_std,3
3,ECG,ECG_51,ecg_raw_variance,4


## Construcción/reutilización de caché por sujeto

Se comprueban nombres, dimensiones, presencia de ambas clases, orden y ausencia de duplicados. Un archivo parcial jamás se declara válido: solo se crea el marcador `manifest.json` una vez completada satisfactoriamente la escritura de sus 455 filas. Si una ejecución se interrumpe, se elimina el archivo temporal antes de reintentar. **No se abre ningún archivo del conjunto de prueba.**

In [3]:
def checkpoint_file(subject):
    assert subject in TRAIN_SUBJECTS, 'Bloqueado: solo entrenamiento.'
    return FULL_DIR / f'{subject}_features_full.pkl'

def cache_paths(subject):
    dest = CACHE_DIR / subject
    return dest, dest/'X_features_first.npy', dest/'y.npy', dest/'manifest.json'

def prepare_subject(subject):
    dest, xp, yp, mp = cache_paths(subject)
    if mp.exists() and xp.exists() and yp.exists():
        m = json.loads(mp.read_text(encoding='utf-8'))
        if m['features']==FEATURE_NAMES and m['n_windows']==len(np.load(yp,mmap_mode='r')):
            xchk=np.load(xp,mmap_mode='r')
            if xchk.shape==(455,m['n_windows']):
                del xchk
                return {'sujeto':subject,'ventanas':m['n_windows'],'cache':'REUTILIZADA'}
    src=checkpoint_file(subject)
    if not src.exists(): raise FileNotFoundError(f'Falta {src}')
    dest.mkdir(parents=True,exist_ok=True)
    for p in (mp, xp, yp):
        if p.exists():p.unlink()
    obj=None; X=None; xout=None
    try:
        print(f'Leyendo {subject}: {src.name} ...',flush=True)
        with open(src,'rb') as f:obj=pickle.load(f)
        assert obj.get('subject')==subject
        block=obj['feature_matrix']
        X=np.asarray(block['X']); names=list(block['names'])
        y=np.asarray(obj['target']).reshape(-1).astype(np.int8,copy=False)
        assert X.ndim==2 and X.shape==(len(y),len(names)),f'{subject}: forma incorrecta'
        assert len(set(names))==len(names),f'{subject}: columnas duplicadas'
        diff=FEATURE_SET-set(names)
        assert not diff,f'{subject}: faltan {len(diff)} variables, ej. {sorted(diff)[:10]}'
        assert set(np.unique(y)).issubset({0,1}) and set(np.unique(y))=={0,1}
        indices=np.array([names.index(k) for k in FEATURE_NAMES],dtype=int)
        xout=np.lib.format.open_memmap(xp,mode='w+',dtype=CACHE_DTYPE,shape=(455,len(y)))
        for j in range(0,455,CACHE_BLOCK_FEATURES):
            inds=indices[j:j+CACHE_BLOCK_FEATURES]
            xout[j:j+len(inds),:]=np.asarray(X[:,inds],dtype=CACHE_DTYPE).T
        xout.flush()
        np.save(yp,y)
        windows=len(y)
        del xout; xout=None
        mp.write_text(json.dumps({'subject':subject,'n_windows':windows,'features':FEATURE_NAMES,
                                 'checkpoint':src.name},ensure_ascii=False),encoding='utf-8')
        return {'sujeto':subject,'ventanas':windows,'cache':'CONSTRUIDA'}
    except Exception:
        for path in (mp,xp,yp):
            if path.exists():path.unlink()
        raise
    finally:
        del obj, X, xout
        gc.collect()

res=[]
for subject in TRAIN_SUBJECTS:
    res.append(prepare_subject(subject))
    gc.collect()
resumen_cache=pd.DataFrame(res)
display(resumen_cache)
assert resumen_cache['sujeto'].tolist()==TRAIN_SUBJECTS
print('✓ 10 sujetos listos. Los 5 de test no se han abierto.')

,sujeto,ventanas,cache
0,S2,7767,REUTILIZADA
1,S3,7903,REUTILIZADA
2,S4,7943,REUTILIZADA
3,S5,8151,REUTILIZADA
4,S6,8091,REUTILIZADA
5,S7,8075,REUTILIZADA
6,S8,8118,REUTILIZADA
7,S9,8071,REUTILIZADA
8,S10,8391,REUTILIZADA
9,S11,8195,REUTILIZADA


✓ 10 sujetos listos. Los 5 de test no se han abierto.


## Extracción de **una única feature** de los diez participantes

La función siguiente reúne valores y etiquetas de los diez sujetos para la característica solicitada. La identidad individual se conserva para el contraste pareado. Las series completas no se guardan simultáneamente durante el barrido de las 455 características; en cada iteración se cierran sus referencias y se ejecuta recolección de basura.

In [4]:
def load_one_feature(feature_name):
    if feature_name not in NAME_TO_CODE:raise KeyError(feature_name)
    k=FEATURE_NAMES.index(feature_name)
    groups=[]
    for subject in TRAIN_SUBJECTS:
        _,xp,yp,_=cache_paths(subject)
        mmap=np.load(xp,mmap_mode='r')
        values=np.array(mmap[k,:],dtype=np.float32,copy=True)
        del mmap
        y=np.load(yp,mmap_mode='r')  # labels tiny compared with full 455-feature matrix
        y=np.array(y,dtype=np.int8,copy=True)
        assert len(values)==len(y)
        groups.append((subject,values,y))
    return groups

def finite_by_class(groups):
    for s,x,y in groups:
        fin=np.isfinite(x)
        yield s,x[fin & (y==0)],x[fin & (y==1)],int((~fin).sum()),len(x)

## Estadísticas y contraste inferencial

**Descriptivos por ventana:** N, media, desviación estándar muestral, coeficiente de variación `100·DE/|media|` (solo si la media está suficientemente lejos de cero), mínimo, cuartiles Q1/Q2/Q3, máximo e IQR. Se excluyen `NaN` e infinitos **únicamente para ese resumen**; también se informa el porcentaje de valores no finitos.

**Contraste principal por sujeto, no por ventana:** para cada participante se calcula la **mediana** en estrés y en no estrés; se define la diferencia pareada `d_s = mediana(estrés_s) − mediana(no_estrés_s)`. Bajo la hipótesis nula de ausencia de cambio sistemático entre condiciones, se calcula una prueba bilateral exacta de **inversión de signos** del estadístico media de `d_s` usando las `2^n` inversiones posibles (`n≤10`). La prueba presupone intercambiabilidad aproximada de los signos de las diferencias bajo H0, condición que puede verse afectada por orden experimental, mezcla baseline/amusement y variación temporal; sus resultados son exploratorios, no causales.

- **H₀:** las diferencias pareadas de medianas no muestran un desplazamiento sistemático entre condiciones.
- **H₁:** existe un desplazamiento sistemático entre condiciones.
- **p exacto:** proporción de inversiones de signos con estadístico al menos tan extremo como el observado, sin aproximaciones basadas en el número de ventanas.
- **q-FDR:** corrección Benjamini–Hochberg entre **las 455 comparaciones**, contando solo p válidos. q<0,05 representa evidencia exploratoria corregida, no capacidad predictiva garantizada.
- **Tamaño del efecto:** mediana y media de diferencias por sujeto; proporción neta de signos `(N_pos−N_neg)/n` (dirección, no Cliff’s delta tradicional).
- **Separación complementaria:** distancia Wasserstein entre las diez medianas por clase y AUC *aparente* entre medianas de sujetos (diagnóstico **no validado**; no confundir con AUC predictiva en test). Los histogramas superpuestos describen distribución; no se calcula un p KS ingenuo de ventanas.

**Importante:** al mezclar baseline y amusement, los grupos tienen composición temporal diferente. Una diferencia estadística podría reflejar otros cambios de condición o tiempo; no identifica un biomarcador causal del estrés.

In [5]:
from scipy.stats import rankdata

def describe(x):
    z=np.asarray(x,dtype=np.float64);z=z[np.isfinite(z)]
    if len(z)==0:return {'Cantidad':0,**{k:np.nan for k in ['Media','Desviacion','Coeficiente_Variacion','Minimo','Q1','Q2(Mediana)','Q3','Maximo','IQR']}}
    q1,q2,q3=np.quantile(z,[.25,.5,.75]); mu=z.mean();sd=z.std(ddof=1) if len(z)>1 else np.nan
    return dict(Cantidad=len(z),Media=mu,Desviacion=sd,
                Coeficiente_Variacion=(100*sd/abs(mu) if abs(mu)>1e-10 and np.isfinite(sd) else np.nan),
                Minimo=z.min(),Q1=q1,**{'Q2(Mediana)':q2},Q3=q3,Maximo=z.max(),IQR=q3-q1)

def exact_paired_signflip(d):
    d=np.asarray(d,dtype=np.float64);d=d[np.isfinite(d)]
    n=len(d)
    if n<3:return np.nan
    obs=abs(d.mean())
    # 2**10=1024; exact enumeration no random number generator.
    signs=2*np.array(list(product([0,1],repeat=n)),dtype=np.int8)-1
    null=np.abs((signs*d).mean(axis=1))
    return float(np.mean(null>=obs-1e-12*max(1,obs)))

def apparent_auc(subject0,subject1):
    a=np.asarray(subject0,dtype=float);b=np.asarray(subject1,dtype=float)
    if len(a)==0 or len(b)==0:return np.nan
    y=np.r_[np.zeros(len(a)),np.ones(len(b))];x=np.r_[a,b]
    ranks=rankdata(x)
    return float((ranks[len(a):].sum()-len(b)*(len(b)+1)/2)/(len(a)*len(b)))

def analyze_one(feature_name):
    groups=load_one_feature(feature_name)
    parts=[];medians=[]; invalid=0; total=0
    for s,x0,x1,n_bad,n_all in finite_by_class(groups):
        parts.extend([x0,x1]);invalid+=n_bad;total+=n_all
        medians.append((s,np.median(x0) if len(x0) else np.nan,
                            np.median(x1) if len(x1) else np.nan,len(x0),len(x1)))
    per=pd.DataFrame(medians,columns=['sujeto','mediana_no_estres','mediana_estres','n0','n1'])
    d=(per.mediana_estres-per.mediana_no_estres).dropna().to_numpy(dtype=float)
    ok=per[['mediana_no_estres','mediana_estres']].notna().all(axis=1)
    ma=per.loc[ok,'mediana_no_estres'].to_numpy();mb=per.loc[ok,'mediana_estres'].to_numpy()
    res={'codigo':NAME_TO_CODE[feature_name],'feature_name':feature_name,
         'n_sujetos_pareados':len(d),'n_ventanas':total,'n_no_finitos':invalid,
         'porcentaje_no_finitos':100*invalid/total if total else np.nan,
         'p_pareada_exacta':exact_paired_signflip(d),'diferencia_mediana_pareada':np.median(d) if len(d) else np.nan,
         'diferencia_media_pareada':np.mean(d) if len(d) else np.nan,
         'dominancia_signos_pareados':(np.count_nonzero(d>0)-np.count_nonzero(d<0))/len(d) if len(d) else np.nan,
         'wasserstein_median_sujeto':wasserstein_distance(ma,mb) if len(d) else np.nan,
         'auc_aparente_median_sujeto':apparent_auc(ma,mb) if len(d) else np.nan}
    # full-window descriptive (memory: only ONE feature; no giant dataframe).
    values0=np.concatenate(parts[0::2]);values1=np.concatenate(parts[1::2]);values=np.r_[values0,values1]
    stats={label:describe(v) for label,v in [('Total',values),('No estrés (0)',values0),('Estrés (1)',values1)]}
    del values,values0,values1,parts,groups
    gc.collect()
    return res,stats,per

def bh_fdr(p):
    p=np.asarray(p,dtype=float);q=np.full(p.shape,np.nan)
    idx=np.where(np.isfinite(p))[0];m=len(idx)
    if not m:return q
    ordered=idx[np.argsort(p[idx])]
    v=p[ordered]*m/np.arange(1,m+1)
    q[ordered]=np.clip(np.minimum.accumulate(v[::-1])[::-1],0,1)
    return q

## Primera pasada: resultados estadísticos con reutilización controlada

Si los tres CSV estadísticos existen y contienen exactamente las 455 características, los códigos actuales, los diez sujetos de entrenamiento y las columnas necesarias, se **recuperan sin recalcular**. Si están incompletos o son incompatibles, se recalcula el análisis estadístico. No se usa información de S13–S17.

La reanudación de figuras de la siguiente etapa opera **independientemente**: aunque falten imágenes, se pueden recuperar estadísticas de los CSV. Los CSV deben proceder del mismo procesamiento; si cambiaste checkpoints, criterio de selección o cálculos estadísticos, establece `REUSE_STATS_CSV = False` para recalcular.


In [6]:
# Reutilización de CSV con verificación de integridad y esquema
STATS_FILES = {
    'tests': OUT_DIR / 'resumen_estadistico_455.csv',
    'descriptivos': OUT_DIR / 'descriptivos_455_clases.csv',
    'pareados': OUT_DIR / 'medianas_por_sujeto_y_clase.csv',
}

def cargar_estadisticas_si_validas():
    if not REUSE_STATS_CSV or not all(p.is_file() for p in STATS_FILES.values()):
        return None
    try:
        t = pd.read_csv(STATS_FILES['tests'], encoding='utf-8-sig')
        d = pd.read_csv(STATS_FILES['descriptivos'], encoding='utf-8-sig')
        p = pd.read_csv(STATS_FILES['pareados'], encoding='utf-8-sig')
        required_t = {'codigo','feature_name','p_pareada_exacta','q_FDR_BH','significativa_FDR_0_05',
                      'n_sujetos_pareados','diferencia_mediana_pareada','dominancia_signos_pareados',
                      'wasserstein_median_sujeto','auc_aparente_median_sujeto','n_no_finitos','porcentaje_no_finitos'}
        required_d = {'codigo','feature_name','grupo','Cantidad','Media','Desviacion',
                      'Coeficiente_Variacion','Minimo','Q1','Q2(Mediana)','Q3','Maximo','IQR'}
        required_p = {'codigo','feature_name','sujeto','mediana_no_estres','mediana_estres','n0','n1'}
        assert required_t.issubset(t.columns) and required_d.issubset(d.columns) and required_p.issubset(p.columns)
        assert len(t) == len(FEATURE_NAMES) == 455
        assert t.feature_name.tolist() == FEATURE_NAMES
        assert t.codigo.tolist() == [NAME_TO_CODE[f] for f in FEATURE_NAMES]
        assert len(d)==3*455 and len(p)==len(TRAIN_SUBJECTS)*455
        assert set(d.grupo)=={'Total','No estrés (0)','Estrés (1)'}
        assert d.groupby('feature_name')['grupo'].nunique().eq(3).all()
        assert set(p.sujeto)==set(TRAIN_SUBJECTS)
        assert p.groupby('feature_name')['sujeto'].nunique().eq(10).all()
        assert set(d.feature_name)==set(FEATURE_NAMES)==set(p.feature_name)
        q_calculado=bh_fdr(t.p_pareada_exacta.to_numpy())
        assert np.allclose(q_calculado,t.q_FDR_BH.to_numpy(),equal_nan=True,rtol=1e-5,atol=1e-8)
        # CSV de pandas puede leer bool como booleanos; forzar interpretación limpia.
        t['significativa_FDR_0_05']=t['q_FDR_BH']<0.05
        return t,d,p
    except Exception as exc:
        print(f'Los CSV no se reutilizarán porque no pasaron validación: {exc}')
        return None

recuperado = cargar_estadisticas_si_validas()
if recuperado is not None:
    tests, stats_all, paired_all = recuperado
    print('✓ Estadísticas recuperadas de CSV: 455 características; 10 sujetos; sin repetir cálculos.')
else:
    resultados=[];descriptivos=[];sujetos_resumen=[]
    for j,name in enumerate(FEATURE_NAMES,1):
        stat,table,per=analyze_one(name)
        resultados.append(stat)
        for clase,values in table.items():
            descriptivos.append({'codigo':NAME_TO_CODE[name],'feature_name':name,'grupo':clase,**values})
        for row in per.to_dict('records'):
            sujetos_resumen.append({'codigo':NAME_TO_CODE[name],'feature_name':name,**row})
        if j%25==0 or j==455: print(f'{j}/455 características analizadas',flush=True)
    tests=pd.DataFrame(resultados)
    tests['q_FDR_BH']=bh_fdr(tests['p_pareada_exacta'])
    tests['significativa_FDR_0_05']=tests['q_FDR_BH']<0.05
    stats_all=pd.DataFrame(descriptivos)
    paired_all=pd.DataFrame(sujetos_resumen)
    tests.to_csv(STATS_FILES['tests'],index=False,encoding='utf-8-sig')
    stats_all.to_csv(STATS_FILES['descriptivos'],index=False,encoding='utf-8-sig')
    paired_all.to_csv(STATS_FILES['pareados'],index=False,encoding='utf-8-sig')
    del resultados, descriptivos, sujetos_resumen
    gc.collect()
    print('✓ Estadísticas calculadas y guardadas; 455 características.')

print('Total:',len(tests),'| p válidos:',tests.p_pareada_exacta.notna().sum(),
      '| q-FDR < 0,05:',tests.significativa_FDR_0_05.sum())
display(tests.head(8))


✓ Estadísticas recuperadas de CSV: 455 características; 10 sujetos; sin repetir cálculos.
Total: 455 | p válidos: 451 | q-FDR < 0,05: 261


,codigo,feature_name,n_sujetos_pareados,n_ventanas,n_no_finitos,porcentaje_no_finitos,p_pareada_exacta,diferencia_mediana_pareada,diferencia_media_pareada,dominancia_signos_pareados,wasserstein_median_sujeto,auc_aparente_median_sujeto,q_FDR_BH,significativa_FDR_0_05
0,ECG_15,ecg_raw_mean,10,80705,0,0.0,0.001953,-0.000107,-0.000175,-1.0,0.000175,0.36,0.005151,True
1,ECG_20,ecg_raw_median,10,80705,0,0.0,0.492188,-0.001991,-0.002842,-0.2,0.008218,0.54,0.579573,False
2,ECG_49,ecg_raw_std,10,80705,0,0.0,0.021484,0.033987,0.032420,0.4,0.039523,0.69,0.037702,True
3,ECG_51,ecg_raw_variance,10,80705,0,0.0,0.015625,0.019651,0.019447,0.4,0.021808,0.69,0.028530,True
4,ECG_22,ecg_raw_min,10,80705,0,0.0,0.005859,-0.144447,-0.213332,-0.8,0.213332,0.28,0.012234,True
5,ECG_14,ecg_raw_max,10,80705,0,0.0,0.503906,0.000092,0.010652,0.7,0.018791,0.65,0.588761,False
6,ECG_35,ecg_raw_range,10,80705,0,0.0,0.003906,0.171501,0.229408,0.8,0.229408,0.69,0.008552,True
7,ECG_27,ecg_raw_q01,10,80705,0,0.0,0.089844,-0.058596,-0.075400,-0.4,0.080939,0.37,0.133288,False


## Visualización integrada por característica: seis paneles, una imagen

Se combinan en una figura de alta resolución: (1) histograma de todas las ventanas de entrenamiento; (2) violín + caja con atípicos según Tukey (1,5 × IQR); (3) histogramas superpuestos con transparencia separados por target; (4) violines y cajas por target con valores fuera de los bigotes; (5) tabla univariada completa y descriptivos de ambas clases; (6) hipótesis, p exacto, q-FDR, efecto, Wasserstein, AUC aparente y advertencia sobre dependencia.

**Sobre los outliers:** un punto fuera de `Q1−1,5·IQR` o `Q3+1,5·IQR` es un atípico **estadístico**, no necesariamente un error biomédico ni una observación que deba eliminarse. Para claridad, el boxplot calcula esos límites sobre todas las ventanas y la nube dibujada está limitada por sujeto y clase; se reporta el porcentaje real de atípicos en la figura. Los violines ilustran densidad suavizada a partir de una submuestra reproducible, mientras que los cuartiles y bigotes provienen de todas las observaciones.

**Lectura de las figuras.** El título general indica «10 Sujetos Train» y el subtítulo enumera cada participante: S2, S3, S4, S5, S6, S7, S8, S9, S10 y S11. El panel 6 utiliza una prueba bilateral exacta de inversión de signos basada en la media de las diez diferencias pareadas de medianas por sujeto, no un contraste ingenuo entre ventanas solapadas. Con diez sujetos hay 2¹⁰ = 1024 configuraciones de signos. La decisión se hace **solo con q-FDR (Benjamini–Hochberg) < 0,05** al corregir todas las variables, no con p sin corregir; un resultado significativo no garantiza utilidad predictiva.


In [7]:
from matplotlib.gridspec import GridSpec

COLORS={0:'#2c8f93',1:'#d76b50'}

def sampled(x,max_n,seed):
    x=np.asarray(x)
    if len(x)<=max_n:return x
    return np.random.default_rng(seed).choice(x,size=max_n,replace=False)

def plot_violin_box(ax,data,labels,colors,title):
    positions=np.arange(1,len(data)+1)
    clean=[np.asarray(v)[np.isfinite(v)] for v in data]
    valid=[(pos,vals,colors[i]) for i,(pos,vals) in enumerate(zip(positions,clean)) if len(vals)>1]
    if valid:
        vpos=[it[0] for it in valid]; vdata=[sampled(it[1],8000,SEED+i) for i,it in enumerate(valid)]
        v=ax.violinplot(vdata,positions=vpos,widths=.67,showmeans=False,showextrema=False)
        for part,color in zip(v['bodies'],[it[2] for it in valid]):
            part.set_facecolor(color);part.set_alpha(.35);part.set_edgecolor(color)
        box=ax.boxplot([it[1] for it in valid],positions=vpos,widths=.22,whis=1.5,
                       showfliers=False,patch_artist=True,manage_ticks=False,
                       medianprops={'color':'black','linewidth':1.8})
        for patch in box['boxes']:
            patch.set_facecolor('white');patch.set_alpha(.93)
        for i,(pos,vals,col) in enumerate(valid):
            q1,q3=np.quantile(vals,[.25,.75]);iqr=q3-q1
            out=vals[(vals<q1-1.5*iqr)|(vals>q3+1.5*iqr)]
            dots=sampled(out,180,SEED+i+21)
            if len(dots):
                ax.scatter(np.full(len(dots),pos)+np.random.default_rng(SEED+i).uniform(-.085,.085,len(dots)),
                           dots,s=8,alpha=.40,color=col,zorder=3)
            pct=100*len(out)/len(vals)
            ax.text(pos,.012,f'Atípicos: {pct:.1f}%',transform=ax.get_xaxis_transform(),
                    ha='center',va='bottom',fontsize=11,color=col)
    ax.set_xticks(positions,labels);ax.set_title(title,loc='left',fontweight='bold',pad=12)
    ax.grid(axis='y',alpha=.2)

def draw_stat_table(ax,stats):
    ax.axis('off');ax.set_title('5 · Estadísticos descriptivos por ventana',loc='left',fontweight='bold',pad=16)
    col=['Grupo','Cantidad','Media','Desv.','CV (%)','Mín.','Q1','Mediana','Q3','Máx.','IQR']
    def fmt(k,v):
        if k=='Cantidad':return f'{int(v):,}'
        return f'{v:.3g}' if np.isfinite(v) else '—'
    vals=[]
    for label,v in stats.items():
        vals.append([label]+[fmt(k,v[k]) for k in ['Cantidad','Media','Desviacion','Coeficiente_Variacion','Minimo','Q1','Q2(Mediana)','Q3','Maximo','IQR']])
    tb=ax.table(cellText=vals,colLabels=col,loc='upper center',cellLoc='center',colLoc='center',
                bbox=[0,.30,1,.58])
    tb.auto_set_font_size(False);tb.set_fontsize(15.5);tb.auto_set_column_width(col=list(range(len(col))))
    for (r,c),cell in tb.get_celld().items():
        cell.set_edgecolor('#e3e8ed');cell.set_facecolor('#edf2f7' if r==0 else ('#f7f9fa' if r%2==1 else 'white'))
        if r==0:cell.get_text().set_weight('bold')
    ax.text(0,.20,'El CV se omite cuando la media es ≈ 0; no debe interpretarse en señales con media cercana a cero.',
            fontsize=15.5,va='top',transform=ax.transAxes)
    ax.text(0,.08,'Cuartiles, IQR, extremos y atípicos se calculan con todas las ventanas finitas de los 10 sujetos.',
            fontsize=15.5,va='top',transform=ax.transAxes)

def draw_tests(ax,row):
    ax.axis('off')
    ax.set_title('6 · Prueba pareada por sujeto y decisión estadística',loc='left',fontweight='bold',pad=18)
    p=float(row['p_pareada_exacta'])
    q=float(row['q_FDR_BH'])
    n=int(row['n_sujetos_pareados'])
    signif=bool(np.isfinite(q) and q<0.05)
    def fmt(x,precision=4):
        return f'{x:.{precision}g}' if np.isfinite(x) else '—'
    if not np.isfinite(p):
        conclusion='NO EVALUABLE: faltan sujetos con ambas clases válidas.'
        color='#a34a21'
    elif signif:
        conclusion='SÍ: diferencia estadísticamente significativa tras FDR.'
        color='#08776e'
    else:
        conclusion='NO: no se detecta diferencia estadísticamente significativa tras FDR.'
        color='#a34a21'

    lines=[
        'Unidad inferencial: sujeto (no ventana solapada).',
        'Para cada sujeto se calcula d = mediana(estrés 1) − mediana(no estrés 0).',
        'Estadístico contrastado: magnitud de la media observada de d.',
        'Prueba: inversión exacta bilateral de signos; no es una prueba por rangos.',
        f'Con {n} sujetos pareados se evalúan 2^{n} = {2**n:,} combinaciones de signos.',
        f'p exacta = {fmt(p,5)}  |  q-FDR (BH) = {fmt(q,5)}  |  Sujetos pareados = {n}/10',
        'Regla de decisión: hay significancia solo si q-FDR < 0,05',
        conclusion,
        f'Dirección: mediana de diferencias = {fmt(row["diferencia_mediana_pareada"],5)}.',
        f'Dominancia de signos = {fmt(row["dominancia_signos_pareados"],4)}  |  Wasserstein = {fmt(row["wasserstein_median_sujeto"],5)}',
        f'AUC aparente (medianas/sujeto, no validada) = {fmt(row["auc_aparente_median_sujeto"],4)}  |  No finitos = {int(row["n_no_finitos"]):,} ({row["porcentaje_no_finitos"]:.2f}%).'
    ]
    ys=[0.93,0.84,0.75,0.66,0.57,0.48,0.39,0.29,0.20,0.12,0.04]
    for i,(s,y) in enumerate(zip(lines,ys)):
        if i==7:
            ax.text(.01,y,s,transform=ax.transAxes,ha='left',va='top',fontsize=17.5,weight='bold',color=color)
        elif i in (5,6):
            ax.text(.01,y,s,transform=ax.transAxes,ha='left',va='top',fontsize=16.6,weight='bold',color='#263244')
        else:
            ax.text(.01,y,s,transform=ax.transAxes,ha='left',va='top',fontsize=16.2,color='#263244')

def plot_one(name,stat,stats,save=True,show=False):
    groups=load_one_feature(name)
    raw0=[];raw1=[];scatter0=[];scatter1=[]
    for k,(s,x0,x1,_,_) in enumerate(finite_by_class(groups)):
        raw0.append(x0);raw1.append(x1)
        scatter0.append(sampled(x0,MAX_POINTS_PER_SUBJECT_CLASS,SEED+k))
        scatter1.append(sampled(x1,MAX_POINTS_PER_SUBJECT_CLASS,SEED+100+k))
    vals0=np.concatenate(raw0);vals1=np.concatenate(raw1)
    view0=np.concatenate(scatter0);view1=np.concatenate(scatter1)
    view=np.r_[view0,view1]
    fig=plt.figure(figsize=(26,20),constrained_layout=False)
    gs=GridSpec(3,2,figure=fig,height_ratios=[1,1,1.40],hspace=.38,wspace=.19,left=.055,right=.975,bottom=.06,top=.86)
    axs=[fig.add_subplot(gs[r,c]) for r in range(3) for c in range(2)]
    title=f'Análisis univariado y bivariado Feature {NAME_TO_CODE[name]} · {name} sobre 10 Sujetos Train'
    fig.suptitle(title,fontsize=27,fontweight='bold',y=.972)
    fig.text(.5,.898,'Participantes de entrenamiento: '+', '.join(TRAIN_SUBJECTS)+'  |  Test reservado: '+', '.join(TEST_RESERVED)+' (no analizado)',ha='center',va='center',fontsize=16.5,color='#334155')
    if len(view):
        axs[0].hist(view,bins=42,color='#5479a0',alpha=.82,density=True)
    axs[0].set_title('1 · Univariado | Histograma global (10 sujetos)',loc='left',fontweight='bold');axs[0].set_ylabel('Densidad');axs[0].set_xlabel('Valor de la característica')
    plot_violin_box(axs[1],[np.r_[vals0,vals1]],['10 sujetos Train'],['#5479a0'],
                    '2 · Univariado | Violín + boxplot (atípicos 1,5 × IQR)')
    if len(view0) and len(view1):
        edges=np.histogram_bin_edges(np.r_[view0,view1],bins=42)
        for c,sub,label in [(0,view0,'No estrés (0)'),(1,view1,'Estrés (1)')]:
            axs[2].hist(sub,bins=edges,density=True,alpha=.47,color=COLORS[c],label=label)
        axs[2].legend(fontsize=12,frameon=False)
    axs[2].set_title('3 · Bivariado | Histogramas superpuestos por target',loc='left',fontweight='bold');axs[2].set_ylabel('Densidad');axs[2].set_xlabel('Valor de la característica')
    plot_violin_box(axs[3],[vals0,vals1],['No estrés (0)','Estrés (1)'],[COLORS[0],COLORS[1]],
                    '4 · Bivariado | Violines + cajas y atípicos por clase')
    draw_stat_table(axs[4],stats)
    draw_tests(axs[5],stat)
    if save:
        # Escribir a un archivo temporal y hacer rename atómico.
        # Si se interrumpe el proceso, el PNG definitivo nunca queda parcialmente sobrescrito.
        target = FIG_DIR / f'{NAME_TO_CODE[name]}__{name}.png'
        temp = FIG_DIR / f'{NAME_TO_CODE[name]}__{name}.tmp.png'
        try:
            fig.savefig(temp,dpi=EXPORT_DPI,bbox_inches='tight',pad_inches=.35)
            os.replace(temp,target)
        finally:
            if temp.exists(): temp.unlink()
    if show:display(fig)
    plt.close(fig)
    del fig,axs,gs,groups,raw0,raw1,scatter0,scatter1,vals0,vals1,view0,view1,view
    gc.collect()


## Segunda pasada: completar imágenes pendientes (reanudación)

Esta etapa **no vuelve a crear las imágenes válidas** ya existentes en `FIG_DIR`. Para cada una de las 455 características se verifica el PNG correspondiente por **código y nombre exacto**. Los PNG inexistentes, corruptos o demasiado pequeños se reconstruyen; los existentes se conservan. Cada imagen nueva se guarda primero con extensión temporal y se renombra al finalizar correctamente para prevenir archivos truncados.

`FIGURE_LIMIT = None` significa *revisar las 455 características, pero generar solo las pendientes*. `FIGURE_LIMIT = 5` limita la revisión a las cinco primeras. `RESUME_FIGURES = False` fuerza la regeneración completa si algún día cambias el diseño. Las estadísticas y la caché de los sujetos se reutilizan cuando son válidas. **No borres `OUT_DIR`, `CACHE_DIR` ni `FIG_DIR`.**


In [8]:
# Verificación de PNG + reanudación de figuras sin recalcular las existentes.
from PIL import Image, UnidentifiedImageError

def ruta_figura(nombre):
    return FIG_DIR / f'{NAME_TO_CODE[nombre]}__{nombre}.png'

def imagen_valida(path):
    if not path.is_file() or path.stat().st_size < 15_000:
        return False
    try:
        with Image.open(path) as im:
            ancho, alto = im.size
            im.verify()  # comprueba que el contenido del PNG no esté truncado
        return ancho >= 1200 and alto >= 900
    except (OSError, ValueError, UnidentifiedImageError):
        return False

assert len(tests)==455 and tests.feature_name.tolist()==FEATURE_NAMES, 'Tabla estadística incompleta/desordenada.'
assert set(stats_all.feature_name)==set(FEATURE_NAMES), 'Descriptivos incompletos.'
test_map=tests.set_index('feature_name')
stat_map={name:{row['grupo']:{key:row[key] for key in ['Cantidad','Media','Desviacion','Coeficiente_Variacion','Minimo','Q1','Q2(Mediana)','Q3','Maximo','IQR']}
                 for _,row in stats_all.loc[stats_all.feature_name==name].iterrows()}
          for name in FEATURE_NAMES}

subset=FEATURE_NAMES[:FIGURE_LIMIT] if FIGURE_LIMIT is not None else FEATURE_NAMES
ya_listas=[name for name in subset if imagen_valida(ruta_figura(name))]
pendientes=[name for name in subset if not imagen_valida(ruta_figura(name))] if RESUME_FIGURES else list(subset)
print(f'Características solicitadas: {len(subset)} | Imágenes completas encontradas: {len(ya_listas)}')
print(f'Imágenes por generar/regenerar: {len(pendientes)}')
if pendientes:
    print('Primeras pendientes:',[(NAME_TO_CODE[n],n) for n in pendientes[:8]])

exitos=0
errores=[]
for i,name in enumerate(pendientes,1):
    try:
        plot_one(name,test_map.loc[name],stat_map[name],save=True,show=bool(SHOW_FIGURES_IN_NOTEBOOK))
        assert imagen_valida(ruta_figura(name)), 'El PNG generado no pasó la comprobación de integridad.'
        exitos+=1
    except Exception as exc:
        errores.append((NAME_TO_CODE[name],name,str(exc)))
        print(f'ERROR en {NAME_TO_CODE[name]} · {name}: {exc}',flush=True)
    finally:
        gc.collect()
    if i%10==0 or i==len(pendientes):
        print(f'Procesadas {i}/{len(pendientes)} pendientes | Generadas: {exitos} | Errores: {len(errores)}',flush=True)

# Verificación FINAL independiente de la cantidad procesada en esta ejecución.
verificadas=[name for name in FEATURE_NAMES if imagen_valida(ruta_figura(name))]
faltantes=[name for name in FEATURE_NAMES if not imagen_valida(ruta_figura(name))]
print('\n=== CONTROL DE COMPLETITUD DE FIGURAS ===')
print(f'Imágenes completas: {len(verificadas)}/455')
print(f'Faltantes/corruptas: {len(faltantes)}')
print(f'Generadas ahora: {exitos}')
print('Ruta:',FIG_DIR)
if faltantes:
    display(pd.DataFrame({'codigo':[NAME_TO_CODE[n] for n in faltantes], 'feature_name':faltantes}))
else:
    print('✓ Las 455 figuras están completas y legibles.')
if errores:
    print('Errores encontrados en esta ejecución:')
    display(pd.DataFrame(errores,columns=['codigo','feature_name','error']))


Características solicitadas: 455 | Imágenes completas encontradas: 405
Imágenes por generar/regenerar: 50
Primeras pendientes: [('RESP_54', 'resp_relative_power_fast_breathing'), ('RESP_62', 'resp_sample_entropy'), ('RESP_1', 'resp_approximate_entropy'), ('RESP_39', 'resp_permutation_entropy'), ('RESP_26', 'resp_lempel_ziv_complexity'), ('RESP_40', 'resp_petrosian_fd'), ('RESP_24', 'resp_katz_fd'), ('RESP_18', 'resp_higuchi_fd')]
Procesadas 10/50 pendientes | Generadas: 10 | Errores: 0
Procesadas 20/50 pendientes | Generadas: 20 | Errores: 0
Procesadas 30/50 pendientes | Generadas: 30 | Errores: 0
Procesadas 40/50 pendientes | Generadas: 40 | Errores: 0
Procesadas 50/50 pendientes | Generadas: 50 | Errores: 0

=== CONTROL DE COMPLETITUD DE FIGURAS ===
Imágenes completas: 455/455
Faltantes/corruptas: 0
Generadas ahora: 50
Ruta: C:\Users\Victor\Documents\THESIS_WESAD\wesad_feature_output\Analisis_univariado_bivariado_train_10_sujetos\figuras_455
✓ Las 455 figuras están completas y legibl

## Informe de calidad y uso responsable de los resultados

Las cifras de significancia no constituyen una política automática de selección. Para construir modelos, cada decisión supervisada sobre features se realizará exclusivamente con entrenamientos internos por participante; S13–S17 permanecerán reservados. Los valores FDR aquí sirven para exploración y formulación de hipótesis: con solo 10 participantes, existe incertidumbre importante; las distribuciones pueden diferir por los protocolos o la combinación baseline/amusement.

Revisar especialmente: proporción de `NaN`/inf, características casi constantes, redundancia entre coeficientes derivados matemáticamente, variabilidad intersujeto y estabilidad de efectos. Los atípicos por IQR no equivalen automáticamente a errores de sensor. La **AUC aparente** de la figura no mide rendimiento de clasificación fuera de muestra. Si todas las ventanas de una misma persona pasan a distintos subconjuntos de ajuste, se produce fuga de información; se requieren particiones agrupadas por sujeto.

In [14]:
print('=== RESUMEN FINAL TRAIN ===')
print('Sujetos usados:',TRAIN_SUBJECTS)
print('Sujetos no abiertos:',TEST_RESERVED)
print('Features:',len(tests),'| ECG 129 | EDA 226 | RESP 100')
print('Variables con q<0.05 (exploratorias):',int(tests['significativa_FDR_0_05'].sum()))
print('Características con no finitos:',int((tests['n_no_finitos']>0).sum()))
print('CSV:',OUT_DIR)
print('Imágenes:',FIG_DIR)

def interpretar_feature(row):
    q = row['q_FDR_BH']
    dif = row['diferencia_mediana_pareada']
    if pd.isna(q):
        return 'No evaluable estadísticamente'
    if q < 0.05:
        resultado = 'Diferencia significativa tras FDR (q < 0.05)'
    else:
        resultado = 'Sin diferencia significativa tras FDR (q ≥ 0.05)'
    if pd.isna(dif):
        direccion = 'dirección no determinada'
    elif dif > 0:
        direccion = 'valores mayores en estrés'
    elif dif < 0:
        direccion = 'valores menores en estrés'
    else:
        direccion = 'sin desplazamiento de mediana'
    return f'{resultado} | {direccion}'

orden = {'ECG': 0, 'EDA': 1, 'RESP': 2}

result_interpreta = (
    tests.assign(
        _grupo=tests['codigo'].str.extract(r'^([A-Za-z]+)')[0].map(orden),
        _numero=tests['codigo'].str.extract(r'(\d+)')[0].astype(int)
    )
    .sort_values(['_grupo', '_numero'])
    [[
        'codigo', 'feature_name', 'n_sujetos_pareados',
        'p_pareada_exacta', 'q_FDR_BH',
        'diferencia_mediana_pareada', 'porcentaje_no_finitos'
    ]]
    .copy()
)
result_interpreta['Interpretacion'] = result_interpreta.apply(interpretar_feature, axis=1)
display(result_interpreta)

print('Figuras finales verificadas:',sum(imagen_valida(ruta_figura(n)) for n in FEATURE_NAMES),'de 455')


=== RESUMEN FINAL TRAIN ===
Sujetos usados: ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11']
Sujetos no abiertos: ['S13', 'S14', 'S15', 'S16', 'S17']
Features: 455 | ECG 129 | EDA 226 | RESP 100
Variables con q<0.05 (exploratorias): 261
Características con no finitos: 5
CSV: C:\Users\Victor\Documents\THESIS_WESAD\wesad_feature_output\Analisis_univariado_bivariado_train_10_sujetos
Imágenes: C:\Users\Victor\Documents\THESIS_WESAD\wesad_feature_output\Analisis_univariado_bivariado_train_10_sujetos\figuras_455


,codigo,feature_name,n_sujetos_pareados,p_pareada_exacta,q_FDR_BH,diferencia_mediana_pareada,porcentaje_no_finitos,Interpretacion
22,ECG_1,ecg_raw_auc_absolute,10,0.029297,0.050049,1.527959e+00,0.000000,Sin diferencia significativa tras FDR (q ≥ 0.0...
23,ECG_2,ecg_raw_auc_signed,10,0.001953,0.005151,-6.419327e-03,0.000000,Diferencia significativa tras FDR (q < 0.05) |...
35,ECG_3,ecg_raw_clearance_factor,10,0.080078,0.121600,-3.867773e+00,0.000000,Sin diferencia significativa tras FDR (q ≥ 0.0...
28,ECG_4,ecg_raw_coefficient_variation,10,0.013672,0.025908,5.871866e+01,0.000000,Diferencia significativa tras FDR (q < 0.05) |...
32,ECG_5,ecg_raw_crest_factor,10,0.107422,0.154291,-6.903734e-01,0.000000,Sin diferencia significativa tras FDR (q ≥ 0.0...
27,ECG_6,ecg_raw_delta,10,0.267578,0.345781,9.269714e-04,0.000000,Sin diferencia significativa tras FDR (q ≥ 0.0...
36,ECG_7,ecg_raw_dominant_frequency,10,0.261719,0.339181,1.538086e+00,0.000000,Sin diferencia significativa tras FDR (q ≥ 0.0...
20,ECG_8,ecg_raw_energy,10,0.015625,0.028530,8.254535e+02,0.000000,Diferencia significativa tras FDR (q < 0.05) |...
34,ECG_9,ecg_raw_impulse_factor,10,0.056641,0.090907,-2.387260e+00,0.000000,Sin diferencia significativa tras FDR (q ≥ 0.0...
15,ECG_10,ecg_raw_iqr,10,0.062500,0.098558,2.639580e-02,0.000000,Sin diferencia significativa tras FDR (q ≥ 0.0...


Figuras finales verificadas: 455 de 455


In [18]:
tabla_interpretaciones = (
    result_interpreta['Interpretacion']
    .value_counts(dropna=False)
    .rename_axis('Interpretacion')
    .to_frame('Cantidad')
)

tabla_interpretaciones['Porcentaje (%)'] = (
    tabla_interpretaciones['Cantidad']
    / len(result_interpreta) * 100
).round(2)

tabla_interpretaciones

,Cantidad,Porcentaje (%)
Interpretacion,,
Diferencia significativa tras FDR (q < 0.05) | valores mayores en estrés,184,40.44
Sin diferencia significativa tras FDR (q ≥ 0.05) | valores mayores en estrés,89,19.56
Diferencia significativa tras FDR (q < 0.05) | valores menores en estrés,77,16.92
Sin diferencia significativa tras FDR (q ≥ 0.05) | valores menores en estrés,76,16.70
Sin diferencia significativa tras FDR (q ≥ 0.05) | sin desplazamiento de mediana,25,5.49
No evaluable estadísticamente,4,0.88


In [19]:
import numpy as np
import pandas as pd

s = result_interpreta['Interpretacion'].fillna('').astype(str)

categorias = (
    'Diferencia significativa',
    'Sin diferencia significativa',
    'No evaluable estadísticamente'
)

grupo = np.select(
    [s.str.startswith(c) for c in categorias],
    categorias,
    default='Otras interpretaciones'
)

tabla_resumen = (
    pd.Series(grupo, index=result_interpreta.index, name='Categoría')
    .value_counts()
    .rename_axis('Categoría')
    .to_frame('Cantidad')
)

tabla_resumen['Porcentaje (%)'] = (
    tabla_resumen['Cantidad'] / len(result_interpreta) * 100
).round(2)

tabla_resumen

,Cantidad,Porcentaje (%)
Categoría,,
Diferencia significativa,261,57.36
Sin diferencia significativa,190,41.76
No evaluable estadísticamente,4,0.88


In [20]:
import numpy as np
import pandas as pd

df = result_interpreta.copy()

# Identificar la señal a partir del código
df['Señal'] = df['codigo'].str.extract(r'^(ECG|EDA|RESP)', expand=False)

# Cantidades por señal e interpretación
tabla_por_senal = (
    df.groupby(['Señal', 'Interpretacion'], dropna=False)
      .size()
      .reset_index(name='Cantidad')
)

# Porcentaje dentro de cada señal
tabla_por_senal['Porcentaje (%)'] = (
    tabla_por_senal['Cantidad']
    / tabla_por_senal.groupby('Señal')['Cantidad'].transform('sum')
    * 100
).round(2)

tabla_por_senal

,Señal,Interpretacion,Cantidad,Porcentaje (%)
0,ECG,Diferencia significativa tras FDR (q < 0.05) |...,40,31.01
1,ECG,Diferencia significativa tras FDR (q < 0.05) |...,11,8.53
2,ECG,Sin diferencia significativa tras FDR (q ≥ 0.0...,2,1.55
3,ECG,Sin diferencia significativa tras FDR (q ≥ 0.0...,50,38.76
4,ECG,Sin diferencia significativa tras FDR (q ≥ 0.0...,26,20.16
5,EDA,Diferencia significativa tras FDR (q < 0.05) |...,108,47.79
6,EDA,Diferencia significativa tras FDR (q < 0.05) |...,38,16.81
7,EDA,No evaluable estadísticamente,2,0.88
8,EDA,Sin diferencia significativa tras FDR (q ≥ 0.0...,15,6.64
9,EDA,Sin diferencia significativa tras FDR (q ≥ 0.0...,32,14.16


In [22]:
import numpy as np
import pandas as pd

df = result_interpreta.copy()

# 1. Identificar señal
df['Señal'] = df['codigo'].str.extract(r'^(ECG|EDA|RESP)', expand=False)

# 2. Agrupar interpretaciones por categoría general
s = df['Interpretacion'].fillna('').astype(str)

categorias = [
    'Diferencia significativa',
    'Sin diferencia significativa',
    'No evaluable estadísticamente'
]

df['Interpretacion agrupada'] = np.select(
    [s.str.startswith(c) for c in categorias],
    categorias,
    default='Otras interpretaciones'
)

# 3. Agrupar por señal, categoría general e interpretación completa
tabla_por_senal = (
    df.groupby(
        ['Señal', 'Interpretacion agrupada'],
        dropna=False
    )
    .size()
    .reset_index(name='Cantidad')
)

# 4. Calcular porcentaje dentro de cada señal
tabla_por_senal['Porcentaje (%)'] = (
    tabla_por_senal['Cantidad']
    / tabla_por_senal.groupby('Señal')['Cantidad'].transform('sum')
    * 100
).round(2)

tabla_por_senal

,Señal,Interpretacion agrupada,Cantidad,Porcentaje (%)
0,ECG,Diferencia significativa,51,39.53
1,ECG,Sin diferencia significativa,78,60.47
2,EDA,Diferencia significativa,146,64.60
3,EDA,No evaluable estadísticamente,2,0.88
4,EDA,Sin diferencia significativa,78,34.51
5,RESP,Diferencia significativa,64,64.00
6,RESP,No evaluable estadísticamente,2,2.00
7,RESP,Sin diferencia significativa,34,34.00
